# 09 — Commenti dei lettori agli articoli ICE del NYT

Analisi descrittiva dei commenti raccolti in `01_dataCollectingAndParsing.ipynb` §6: su quali articoli, quanto e chi commenta, quando il dibattito tra lettori si infiamma (burst di Kleinberg su volume e conflittualità dei thread) e se i commenti scelti dalla redazione (NYT Picks) parlano degli stessi soggetti di quelli preferiti dai lettori, e di quali temi parlano i lettori (BERTopic).

**Input**
- `data/processed/nyt/comments_ice.parquet`: un commento o una risposta per riga (286.399).
- `data/processed/nyt/comments_ice_articles.parquet`: un articolo ICE per riga, con i totali dichiarati dal NYT (2.093).
- `data/processed/nyt/articles_ice.parquet`: metadati degli articoli.

**Pseudonimizzazione**: `user_id` e i nomi dei lettori non escono da §1. Ogni utente diventa `user_key` = `u_` + hash SHA-256 di (sale + `user_id`), con il sale in `.env` (`COMMENTS_PSEUDONYM_SALT`). Tutte le tabelle e i grafici scritti da questo notebook usano solo `user_key`.

**Fuori da questo notebook**: il sentiment dei commenti, che verrà fatto insieme a quello degli articoli con lo stesso modello.

## 0. Setup

In [ ]:
import hashlib
import logging
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

%matplotlib inline

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed" / "nyt"
PLOTS_DIR = PROCESSED_DIR / "plots"

IN_COMMENTS = PROCESSED_DIR / "comments_ice.parquet"
IN_COMMENT_ARTICLES = PROCESSED_DIR / "comments_ice_articles.parquet"
IN_ARTICLES = PROCESSED_DIR / "articles_ice.parquet"

OUT_COMMENTS_PSEUDO = PROCESSED_DIR / "comments_ice_pseudo.parquet"
OUT_ARTICLE_STATS = PROCESSED_DIR / "comments_article_stats.csv"
OUT_THREAD_METRICS = PROCESSED_DIR / "comments_thread_metrics.csv"
OUT_BURSTS = PROCESSED_DIR / "comments_bursts.csv"
OUT_PICKS_ENTITIES = PROCESSED_DIR / "comments_picks_entities.csv"
OUT_COMMENT_TOPICS = PROCESSED_DIR / "comments_topics.parquet"
OUT_COMMENT_TOPIC_INFO = PROCESSED_DIR / "comments_topic_info.csv"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", force=True)
log = logging.getLogger("nyt_comments_analysis")

load_dotenv(BASE_DIR / ".env")
PSEUDONYM_SALT = os.getenv("COMMENTS_PSEUDONYM_SALT")
if not PSEUDONYM_SALT:
    raise RuntimeError("COMMENTS_PSEUDONYM_SALT non trovato in .env")

# Slot categoriali della palette di riferimento, in ordine fisso: un colore per periodo
PERIOD_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
MAIN_COLOR = PERIOD_COLORS[0]
MUTED_COLOR = "#b8b8b0"

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e6e1",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
})


def get_periods(max_year: int) -> list[tuple[str, int, int]]:
    """Stessi periodi dei notebook 02-08: (label, anno iniziale incluso, anno finale escluso)."""
    return [
        ("2003-2016", 2003, 2017),
        ("2017-2020", 2017, 2021),
        ("2021-2024", 2021, 2025),
        ("2025-presente", 2025, max_year + 1),
    ]


def assign_period(years: pd.Series) -> pd.Series:
    periods = get_periods(int(years.max()))
    labels = [label for label, _, _ in periods]
    bins = [periods[0][1]] + [end for _, _, end in periods]
    return pd.cut(years, bins=bins, right=False, labels=labels)


def gini(values) -> float:
    """Indice di Gini (0 = distribuzione uniforme, 1 = tutto su un solo elemento)."""
    x = np.sort(np.asarray(values, dtype=float))
    n = len(x)
    if n == 0 or x.sum() == 0:
        return 0.0
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


def top_share(values, fraction: float) -> float:
    """Quota del totale detenuta dal `fraction` piu' grande degli elementi (es. 0.01 = top 1%)."""
    x = np.sort(np.asarray(values, dtype=float))[::-1]
    k = max(1, int(round(len(x) * fraction)))
    return float(x[:k].sum() / x.sum())

## 1. Caricamento e pseudonimizzazione

`parent_user_key` (a chi risponde un commento) si ricava agganciando `parent_id` al commento padre, invece di usare `parent_user_display_name`: i nomi visualizzati non sono univoci e sono dati personali.

Il Parquet pseudonimizzato (`comments_ice_pseudo.parquet`) è la versione condivisibile della tabella: niente `user_id`, niente nomi. `user_location` resta (testo libero scritto dal lettore): va aggregato prima di pubblicarlo.

In [ ]:
def pseudonymize(user_ids: pd.Series, salt: str) -> pd.Series:
    """user_id -> 'u_' + primi 12 caratteri esadecimali di sha256(sale + id). Stabile a parita' di sale."""
    return user_ids.map(lambda uid: "u_" + hashlib.sha256(f"{salt}{uid}".encode()).hexdigest()[:12])


def load_comments() -> pd.DataFrame:
    df = pd.read_parquet(IN_COMMENTS)
    df["user_key"] = pseudonymize(df["user_id"], PSEUDONYM_SALT)
    if df["user_key"].nunique() != df["user_id"].nunique():
        raise RuntimeError("Collisione di pseudonimi: aumentare la lunghezza dell'hash")

    parent_keys = df.set_index("comment_id")["user_key"]
    df["parent_user_key"] = df["parent_id"].map(parent_keys)

    df = df.drop(columns=["user_id", "user_display_name", "parent_user_display_name"])
    df["year"] = df["article_pub_date"].dt.year
    df["period"] = assign_period(df["year"])
    return df


def load_comment_articles() -> pd.DataFrame:
    df = pd.read_parquet(IN_COMMENT_ARTICLES)
    df["year"] = df["article_pub_date"].dt.year
    df["period"] = assign_period(df["year"])
    return df


comments = load_comments()
comment_articles = load_comment_articles()

missing_parent = comments["is_reply"] & comments["parent_user_key"].isna()
log.info("Commenti: %d, utenti: %d, articoli: %d", len(comments), comments["user_key"].nunique(), len(comment_articles))
log.info("Risposte senza commento padre tra i raccolti: %d", missing_parent.sum())

comments.to_parquet(OUT_COMMENTS_PSEUDO, index=False)
log.info("Scritto %s", OUT_COMMENTS_PSEUDO.relative_to(BASE_DIR))

In [ ]:
comments.groupby("period", observed=True).agg(
    commenti=("comment_id", "size"),
    articoli=("article_id", "nunique"),
    utenti=("user_key", "nunique"),
    risposte_pct=("is_reply", lambda s: round(s.mean() * 100, 1)),
    date_imputate_pct=("created_at_imputed", lambda s: round(s.mean() * 100, 1)),
)

## 2. Quali articoli hanno commenti

Il NYT apre i commenti solo su una parte degli articoli, per scelta editoriale. `has_comments` vale `True` se il NYT dichiara almeno un commento: non si distingue un articolo con i commenti chiusi da uno con i commenti aperti ma vuoti (caso raro sul NYT).

Il volume dei commenti per anno va letto insieme al numero di articoli commentabili: un anno con più commenti può avere solo più articoli aperti.

In [ ]:
by_year = comment_articles.groupby("year").agg(
    articoli_ice=("article_id", "size"),
    articoli_con_commenti=("has_comments", "sum"),
    commenti=("n_comments_collected", "sum"),
)
by_year["pct_con_commenti"] = (by_year["articoli_con_commenti"] / by_year["articoli_ice"] * 100).round(1)
by_year["commenti_per_articolo_commentato"] = (
    by_year["commenti"] / by_year["articoli_con_commenti"].replace(0, np.nan)
).round(0)
by_year

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

axes[0].bar(by_year.index, by_year["articoli_ice"], color=MUTED_COLOR, width=0.8, label="articoli ICE")
axes[0].bar(by_year.index, by_year["articoli_con_commenti"], color=MAIN_COLOR, width=0.8, label="di cui con commenti")
axes[0].set_ylabel("articoli")
axes[0].legend(frameon=False, loc="upper left")
axes[0].set_title("Articoli ICE e articoli con commenti, per anno", loc="left")

axes[1].bar(by_year.index, by_year["commenti"] / 1000, color=MAIN_COLOR, width=0.8)
axes[1].set_ylabel("commenti (migliaia)")
axes[1].set_title("Commenti raccolti, per anno di pubblicazione dell'articolo", loc="left")

fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_coverage_by_year.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Sezioni con almeno 20 articoli ICE: quali aprono i commenti piu' spesso
by_section = comment_articles.groupby("section_name").agg(
    articoli_ice=("article_id", "size"),
    articoli_con_commenti=("has_comments", "sum"),
    commenti=("n_comments_collected", "sum"),
)
by_section = by_section[by_section["articoli_ice"] >= 20]
by_section["pct_con_commenti"] = (by_section["articoli_con_commenti"] / by_section["articoli_ice"] * 100).round(1)
by_section["pct_commenti_totali"] = (by_section["commenti"] / comment_articles["n_comments_collected"].sum() * 100).round(1)
by_section.sort_values("commenti", ascending=False)

## 3. Concentrazione per articolo

Quanto i commenti si concentrano su pochi articoli. Gini e quote del top 1% / 10% sono calcolati sui soli articoli con commenti.

In [ ]:
articles_meta = pd.read_parquet(IN_ARTICLES, columns=["id", "headline", "web_url"]).rename(columns={"id": "article_id"})

article_stats = (
    comments.groupby("article_id")
    .agg(
        commenti=("comment_id", "size"),
        risposte=("is_reply", "sum"),
        utenti=("user_key", "nunique"),
        raccomandazioni=("recommendations", "sum"),
        selezioni_editoriali=("editors_selection", "sum"),
        risposte_giornalisti=("is_reporter_reply", "sum"),
        ore_mediane_dalla_pubblicazione=("hours_since_publication", "median"),
    )
    .reset_index()
    .merge(comment_articles[["article_id", "article_pub_date", "period", "section_name"]], on="article_id", how="left")
    .merge(articles_meta, on="article_id", how="left")
    .sort_values("commenti", ascending=False)
)
article_stats["quota_risposte"] = (article_stats["risposte"] / article_stats["commenti"]).round(3)
article_stats.to_csv(OUT_ARTICLE_STATS, index=False)
log.info("Scritto %s", OUT_ARTICLE_STATS.relative_to(BASE_DIR))

n = article_stats["commenti"]
pd.Series({
    "articoli con commenti": len(n),
    "mediana commenti/articolo": n.median(),
    "media commenti/articolo": round(n.mean(), 1),
    "massimo": n.max(),
    "gini": round(gini(n), 3),
    "quota top 1% articoli": round(top_share(n, 0.01), 3),
    "quota top 10% articoli": round(top_share(n, 0.10), 3),
})

In [ ]:
article_stats.head(15)[["article_pub_date", "section_name", "headline", "commenti", "quota_risposte", "utenti"]]

In [ ]:
def lorenz_points(values) -> tuple[np.ndarray, np.ndarray]:
    x = np.sort(np.asarray(values, dtype=float))
    cum = np.concatenate([[0], np.cumsum(x) / x.sum()])
    return np.linspace(0, 1, len(cum)), cum


fig, ax = plt.subplots(figsize=(5.5, 5))
for color, (label, _, _) in zip(PERIOD_COLORS, get_periods(int(comments["year"].max()))):
    values = article_stats.loc[article_stats["period"] == label, "commenti"]
    if len(values) < 2:
        continue
    px, py = lorenz_points(values)
    ax.plot(px, py, color=color, linewidth=2, label=f"{label} (n={len(values)}, Gini {gini(values):.2f})")
ax.plot([0, 1], [0, 1], color=MUTED_COLOR, linewidth=1, linestyle="--")
ax.set_xlabel("quota cumulata degli articoli (dal meno commentato)")
ax.set_ylabel("quota cumulata dei commenti")
ax.set_title("Concentrazione dei commenti per articolo", loc="left")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_lorenz_by_period.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Conflittualità dei thread e burst del dibattito (Kleinberg)

Obiettivo: trovare i momenti in cui il dibattito tra lettori si infiamma, separando due cose diverse:

- **volume**: tanti commenti in poco tempo;
- **conflitto**: conversazioni che si allungano in catene di risposte e in scambi botta e risposta tra le stesse persone.

### Perché non il campo `depth`

`depth` del NYT vale solo 1 (commento) o 2 (risposta): l'interfaccia mostra le risposte tutte allo stesso livello. Ma `parent_id` punta al commento a cui si risponde davvero, anche quando è a sua volta una risposta. Da lì si ricostruisce la **profondità della catena** (`chain_depth`): 1 = commento, 2 = risposta a un commento, 3 = risposta a una risposta, e così via.

### Le due misure di conflitto

- **Risposte in catena** (`deep`): `chain_depth ≥ 3`. La discussione va avanti oltre il primo giro di risposte.
- **Botta e risposta** (`in_exchange`): risposta di A a B in un articolo in cui anche B risponde ad A. Indica un confronto diretto e ripetuto tra due persone. Le risposte a sé stessi sono escluse.

Sono misure **strutturali**: dicono che si discute, non se si è in disaccordo. Il tono (disaccordo, tossicità) richiede il testo e verrà con il sentiment.

### Inizio delle catene: 2018-04

Prima del 2018 tutte le risposte puntano al commento iniziale del thread: le risposte a risposte compaiono a fine novembre 2017 (1% delle risposte) e si stabilizzano ad aprile 2018 (13%, poi 20-30%). È un cambiamento del sistema dei commenti NYT, non del comportamento dei lettori. Le due serie di conflitto partono quindi da `THREADED_REPLIES_START = 2018-04-01`; il 2003-2016 resta fuori dall'analisi del conflitto.

In [ ]:
THREADED_REPLIES_START = pd.Timestamp("2018-04-01", tz="America/New_York")
DEEP_MIN_DEPTH = 3


def add_chain_depth(df: pd.DataFrame) -> pd.DataFrame:
    """chain_depth risalendo parent_id fino al commento iniziale. Risposte con padre non raccolto = 2."""
    depth = pd.Series(np.where(df["is_reply"], 2, 1), index=df["comment_id"].values)
    has_parent = df["parent_id"].notna() & df["parent_id"].isin(df["comment_id"])
    children = df.loc[has_parent, "comment_id"].values
    parents = df.loc[has_parent, "parent_id"].astype("int64").values
    # propagazione a punto fisso: ogni giro allunga di un livello le catene ancora incomplete
    for _ in range(100):
        new_depth = depth.copy()
        new_depth.loc[children] = depth.loc[parents].values + 1
        if new_depth.equals(depth):
            break
        depth = new_depth
    else:
        raise RuntimeError("chain_depth non converge: ciclo in parent_id?")
    df = df.copy()
    df["chain_depth"] = depth.loc[df["comment_id"]].values
    df["deep"] = df["chain_depth"] >= DEEP_MIN_DEPTH
    return df


def add_exchanges(df: pd.DataFrame) -> pd.DataFrame:
    """in_exchange: risposta A -> B nello stesso articolo in cui esiste anche B -> A."""
    replies = df[df["is_reply"] & df["parent_user_key"].notna() & (df["user_key"] != df["parent_user_key"])]
    pairs = set(zip(replies["article_id"], replies["user_key"], replies["parent_user_key"]))
    df = df.copy()
    df["in_exchange"] = [
        is_reply and isinstance(parent, str) and user != parent and (article, parent, user) in pairs
        for article, user, parent, is_reply in zip(df["article_id"], df["user_key"], df["parent_user_key"], df["is_reply"])
    ]
    return df


comments = add_exchanges(add_chain_depth(comments))
comments["day"] = comments["created_at"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
comments["threaded"] = comments["created_at"] >= THREADED_REPLIES_START

display(comments["chain_depth"].value_counts().sort_index().to_frame("commenti").T)
comments[comments["threaded"]].groupby("period", observed=True).agg(
    commenti=("comment_id", "size"),
    risposte_in_catena_pct=("deep", lambda s: round(s.mean() * 100, 1)),
    botta_e_risposta_pct=("in_exchange", lambda s: round(s.mean() * 100, 1)),
    profondita_max=("chain_depth", "max"),
)

### 4.1 Metriche per articolo

**A cosa serve.** Kleinberg (§4.2-4.5) guarda la conflittualità **nel tempo**: trova i giorni in cui si litiga di più. Questa cella la guarda **per articolo**, indipendentemente da quando è uscito: quali articoli producono le discussioni più lunghe e i confronti più diretti. Non è un passaggio necessario per i burst (che usano le colonne `deep` e `in_exchange` dei singoli commenti) e nessuna cella successiva la usa: è una tabella a parte, che serve a

- confrontare la conflittualità con il sentiment dei commenti (quando ci sarà);
- incrociarla con sezione, frame e topic dell'articolo;
- trovare articoli molto conflittuali che i burst non vedono, perché usciti in giorni già pieni di commenti.

**Cosa calcola.** Una riga per articolo con commenti: profondità media (sulle sole risposte) e massima delle catene, quota di risposte in catena, quota di risposte in botta e risposta, numero di coppie di utenti che si rispondono a vicenda. Salvata in `comments_thread_metrics.csv`; sotto, i 10 articoli con la quota di botta e risposta più alta (almeno 200 commenti, per evitare quote alte su thread minuscoli).

Per gli articoli prima di `THREADED_REPLIES_START` le metriche di catena sono strutturalmente basse e non vanno confrontate con quelle successive (colonna `threaded`).

In [ ]:
def thread_metrics(df: pd.DataFrame) -> pd.DataFrame:
    replies = df[df["is_reply"]]
    exchange_dyads = (
        df[df["in_exchange"]]
        .assign(dyad=lambda x: [tuple(sorted(p)) for p in zip(x["user_key"], x["parent_user_key"])])
        .groupby("article_id")["dyad"].nunique()
    )
    metrics = df.groupby("article_id").agg(
        commenti=("comment_id", "size"),
        risposte=("is_reply", "sum"),
        profondita_max=("chain_depth", "max"),
        risposte_in_catena=("deep", "sum"),
        risposte_botta_e_risposta=("in_exchange", "sum"),
        threaded=("threaded", "all"),
    )
    metrics["profondita_media_risposte"] = replies.groupby("article_id")["chain_depth"].mean().round(2)
    metrics["quota_in_catena"] = (metrics["risposte_in_catena"] / metrics["commenti"]).round(3)
    metrics["quota_botta_e_risposta"] = (metrics["risposte_botta_e_risposta"] / metrics["commenti"]).round(3)
    metrics["coppie_botta_e_risposta"] = exchange_dyads.reindex(metrics.index, fill_value=0)
    return (
        metrics.reset_index()
        .merge(article_stats[["article_id", "article_pub_date", "period", "section_name", "headline"]], on="article_id", how="left")
        .sort_values("article_pub_date")
    )


thread_stats = thread_metrics(comments)
thread_stats.to_csv(OUT_THREAD_METRICS, index=False)
log.info("Scritto %s (%d articoli)", OUT_THREAD_METRICS.relative_to(BASE_DIR), len(thread_stats))

# Articoli piu' conflittuali (solo era con catene, almeno 200 commenti per evitare quote su thread minuscoli)
(
    thread_stats[thread_stats["threaded"] & (thread_stats["commenti"] >= 200)]
    .sort_values("quota_botta_e_risposta", ascending=False)
    .head(10)[["article_pub_date", "section_name", "headline", "commenti", "profondita_max", "quota_in_catena", "quota_botta_e_risposta", "coppie_botta_e_risposta"]]
)

### 4.2 Algoritmo di Kleinberg

Kleinberg (2002), *Bursty and Hierarchical Structure in Streams*. La serie è generata da un automa a stati nascosti: lo stato 0 emette al ritmo di fondo, lo stato *i* a un ritmo *s^i* volte più alto. Salire di un livello costa `γ · ln(T)` (T = numero di unità di tempo), scendere è gratis. La sequenza di stati più probabile si trova con Viterbi; un **burst** è un tratto consecutivo con stato ≥ 1, di livello pari allo stato massimo raggiunto. Livelli possibili: 0-3 (`N_LEVELS`).

Due varianti, una per tipo di misura:

| Serie | Variante | Emissione | Unità | Ritmo di fondo |
|---|---|---|---|---|
| Volume | conteggi | Poisson(commenti nell'unità; λ₀ · s^i) | **settimana** | λ₀ = media settimanale del periodo, settimane vuote incluse |
| Conflitto (catena, botta e risposta) | proporzioni (versione *batched* di Kleinberg) | Binomiale(r_t su d_t; p₀ · s^i) | **giorno** | p₀ = quota del periodo; giorni senza commenti esclusi |

**Perché la settimana per il volume.** Con il giorno ogni giornata con un articolo molto commentato diventa un burst a sé, quasi sempre di 1-2 giorni (numeri in §4.5): l'algoritmo restituisce l'elenco degli articoli più commentati, senza aggregare. Con la settimana emergono periodi di più settimane. La giornata di picco e gli articoli trainanti si recuperano dentro ogni burst. I conteggi giornalieri restano nel controllo di sensibilità.

**Perché il giorno per il conflitto.** La misura è una quota: un giorno con pochi commenti pesa poco nella verosimiglianza binomiale e non basta a far scattare un burst, quindi il giorno non frammenta. Con la settimana una quota alta concentrata in un giorno si diluisce nei giorni normali della stessa settimana.

**Perché per periodo.** Kleinberg misura i burst rispetto a un ritmo di fondo unico. Sul corpus intero il fondo sarebbe dominato dal 2025-2026 (196 mila commenti su 286 mila, circa 10 volte i periodi precedenti): il 2025-presente risulterebbe un unico burst lungo e nel 2003-2024 non emergerebbe quasi nulla. Stimando il fondo separatamente nei 4 periodi già usati nei notebook 02-08, ogni burst è "eccezionale rispetto al suo periodo". Conseguenza da tenere presente: un burst del 2021-2024 e uno del 2025-presente non hanno la stessa intensità assoluta.

**Limiti noti.**
- Nei periodi con pochi articoli commentati (2003-2016: 72 articoli; 2021-2024: 32) il fondo settimanale è vicino a zero e quasi ogni settimana con commenti diventa un burst: per il volume questi periodi sono poco informativi.
- La Poisson assume eventi indipendenti; i commenti arrivano a grappoli (un articolo = migliaia di commenti in 24 ore). Il costo di transizione `γ · ln(T)` è piccolo rispetto alla verosimiglianza di conteggi così grandi, quindi per il volume conta soprattutto `s`, poco `γ` (vedi §4.5).
- Nelle serie del conflitto i giorni senza commenti non ci sono: un burst si spezza dove tra due giorni consecutivi della serie passano più di `MAX_GAP_DAYS = 3` giorni (i thread si chiudono quasi sempre entro 24-72 ore). Senza questo taglio uscivano burst di mesi tra articoli lontani (es. 2019-10 → 2020-02).
- Le date dei commenti prima del 2019 sono in parte imputate (data di approvazione, circa 30 minuti dopo): irrilevante a scala di giorno o settimana.

Implementazione propria (nessuna libreria mantenuta), verificata su dati sintetici con un picco noto nella cella seguente.

In [ ]:
N_LEVELS = 3
KLEINBERG_S = 2.0
KLEINBERG_GAMMA = 1.0


def viterbi_states(costs: np.ndarray, gamma: float) -> np.ndarray:
    """Sequenza di stati a costo minimo. costs: (T, K) costo di emissione; salire costa gamma*ln(T) per livello."""
    n_units, n_states = costs.shape
    up_cost = gamma * np.log(n_units)
    trans = np.array([[max(j - i, 0) * up_cost for j in range(n_states)] for i in range(n_states)])
    total = costs[0] + trans[0]
    back = np.zeros((n_units, n_states), dtype=int)
    for t in range(1, n_units):
        candidates = total[:, None] + trans
        back[t] = candidates.argmin(axis=0)
        total = candidates.min(axis=0) + costs[t]
    states = np.zeros(n_units, dtype=int)
    states[-1] = total.argmin()
    for t in range(n_units - 1, 0, -1):
        states[t - 1] = back[t, states[t]]
    return states


def kleinberg_counts(counts, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA, n_levels: int = N_LEVELS) -> np.ndarray:
    """Variante per conteggi: emissione Poisson con ritmo base * s^i (costo = -log verosimiglianza senza costanti)."""
    counts = np.asarray(counts, dtype=float)
    rates = counts.mean() * s ** np.arange(n_levels + 1)
    costs = rates[None, :] - counts[:, None] * np.log(rates)[None, :]
    return viterbi_states(costs, gamma)


def kleinberg_proportions(relevant, totals, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA, n_levels: int = N_LEVELS) -> np.ndarray:
    """Variante batched di Kleinberg: r_t eventi rilevanti su d_t, emissione binomiale con p0 * s^i (< 1)."""
    r = np.asarray(relevant, dtype=float)
    d = np.asarray(totals, dtype=float)
    p0 = r.sum() / d.sum()
    probs = p0 * s ** np.arange(n_levels + 1)
    probs = probs[probs < 1]
    costs = -(r[:, None] * np.log(probs)[None, :] + (d - r)[:, None] * np.log1p(-probs)[None, :])
    return viterbi_states(costs, gamma)


MAX_GAP_DAYS = 3


def state_runs(states: np.ndarray, index: pd.Index | None = None, max_gap_days: int = MAX_GAP_DAYS) -> list[tuple[int, int, int]]:
    """Tratti consecutivi con stato >= 1: (inizio, fine inclusa, livello massimo).

    Con `index` (date delle unita'), un tratto si spezza dove due unita' consecutive distano piu' di
    max_gap_days: nelle serie del conflitto i giorni senza commenti non ci sono, e senza questo taglio
    lo stato alto "sopravvive" per mesi tra due articoli lontani.
    """
    runs, start = [], None
    # nelle serie settimanali le unita' distano sempre 7 giorni: il taglio scatta solo oltre il passo della serie
    step_days = int(pd.Series(index).diff().min().days) if index is not None and len(index) > 1 else 1
    for t in range(len(states) + 1):
        state = states[t] if t < len(states) else 0
        gap = (
            index is not None and start is not None and t < len(states)
            and (index[t] - index[t - 1]).days > max(max_gap_days, step_days)
        )
        if start is not None and (state == 0 or gap):
            runs.append((start, t - 1, int(max(states[start:t]))))
            start = None
        if state > 0 and start is None:
            start = t
    return runs


# Verifica su dati sintetici: picchi noti in [80, 89] (conteggi) e [120, 129] (proporzioni)
rng = np.random.default_rng(42)
synthetic_counts = rng.poisson(10, 200)
synthetic_counts[80:90] = rng.poisson(40, 10)
synthetic_totals = rng.poisson(200, 200)
synthetic_p = np.where((np.arange(200) >= 120) & (np.arange(200) < 130), 0.3, 0.1)
synthetic_relevant = rng.binomial(synthetic_totals, synthetic_p)
print("conteggi:", state_runs(kleinberg_counts(synthetic_counts)))
print("proporzioni:", state_runs(kleinberg_proportions(synthetic_relevant, synthetic_totals)))

### 4.3 Serie e burst per periodo

Per ogni periodo (articoli assegnati per anno di pubblicazione, come nel resto del progetto):

- **volume**: commenti per settimana (lunedì-domenica, ora di New York) dal primo all'ultimo commento del periodo, settimane vuote incluse;
- **catena** e **botta e risposta**: per ogni giorno con commenti dopo `THREADED_REPLIES_START`, risposte in catena (o in botta e risposta) su commenti totali del giorno.

Per ogni burst: inizio, fine, livello, commenti, quota della misura dentro il burst contro quella del periodo, giorno di picco e i 3 articoli che contribuiscono di più (per il volume: più commenti nel burst; per il conflitto: più risposte in catena / in botta e risposta nel burst).

In [ ]:
SERIES = {
    "volume": {"unit": "W", "relevant": None},
    "catena": {"unit": "D", "relevant": "deep"},
    "botta_e_risposta": {"unit": "D", "relevant": "in_exchange"},
}


def unit_index(days: pd.Series, unit: str) -> pd.Series:
    return days.dt.to_period("W-SUN").dt.start_time if unit == "W" else days


def build_series(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Serie (unita' di tempo -> totali, rilevanti) per un periodo, nel formato atteso da Kleinberg."""
    spec = SERIES[name]
    if spec["relevant"] is not None:
        df = df[df["threaded"]]
    if df.empty:
        return pd.DataFrame(columns=["totali", "rilevanti"])
    units = unit_index(df["day"], spec["unit"])
    grouped = df.groupby(units).agg(totali=("comment_id", "size"))
    grouped["rilevanti"] = df.groupby(units)[spec["relevant"]].sum() if spec["relevant"] else grouped["totali"]
    if spec["unit"] == "W":
        # settimane vuote incluse: per il volume un'assenza di commenti e' un dato, non un buco
        full = pd.date_range(grouped.index.min(), grouped.index.max(), freq="7D")
        grouped = grouped.reindex(full, fill_value=0)
    return grouped


def detect(series: pd.DataFrame, name: str, s: float = KLEINBERG_S, gamma: float = KLEINBERG_GAMMA) -> np.ndarray:
    if len(series) < 2 or series["rilevanti"].sum() == 0:
        return np.zeros(len(series), dtype=int)
    if SERIES[name]["relevant"] is None:
        return kleinberg_counts(series["totali"].values, s=s, gamma=gamma)
    return kleinberg_proportions(series["rilevanti"].values, series["totali"].values, s=s, gamma=gamma)


def describe_burst(df_period: pd.DataFrame, series: pd.DataFrame, name: str, start: int, end: int, level: int) -> dict:
    spec = SERIES[name]
    first = series.index[start]
    last = series.index[end] + (pd.Timedelta(days=6) if spec["unit"] == "W" else pd.Timedelta(0))
    window = df_period[(df_period["day"] >= first) & (df_period["day"] <= last)]
    if spec["relevant"]:
        window = window[window["threaded"]]
    rank_col = spec["relevant"] or "comment_id"
    by_article = window.groupby("article_id")[rank_col].agg("sum" if spec["relevant"] else "size").sort_values(ascending=False)
    top = by_article.head(3)
    titles = article_stats.set_index("article_id").loc[top.index]
    in_window = series.iloc[start:end + 1]
    row = {
        "serie": name,
        "inizio": first.date(),
        "fine": last.date(),
        "unita": end - start + 1,
        "livello": level,
        "commenti": len(window),
        "giorno_di_picco": window.groupby("day").size().idxmax().date() if len(window) else None,
        "articolo_principale": top.index[0] if len(top) else None,
        "articoli_principali": " | ".join(
            f"{t.article_pub_date.date()} {t.headline} ({n})" for (_, t), n in zip(titles.iterrows(), top.values)
        ),
    }
    if spec["relevant"]:
        row["quota_nel_burst"] = round(in_window["rilevanti"].sum() / in_window["totali"].sum(), 3)
        row["quota_periodo"] = round(series["rilevanti"].sum() / series["totali"].sum(), 3)
    return row


period_labels = [label for label, _, _ in get_periods(int(comments["year"].max()))]
series_store, burst_rows = {}, []
for period in period_labels:
    df_period = comments[comments["period"] == period]
    for name in SERIES:
        series = build_series(df_period, name)
        states = detect(series, name)
        series_store[(period, name)] = (series, states)
        for start, end, level in state_runs(states, series.index):
            burst_rows.append({"periodo": period, **describe_burst(df_period, series, name, start, end, level)})

bursts = pd.DataFrame(burst_rows)
bursts.groupby(["serie", "periodo"], observed=True).agg(
    burst=("inizio", "size"), unita_in_burst=("unita", "sum"), commenti_in_burst=("commenti", "sum"), livello_max=("livello", "max")
).unstack("serie")

#### Come leggere i grafici

I tre grafici hanno la stessa struttura: **una riga per periodo** (2003-2016, 2017-2020, 2021-2024, 2025-presente), tempo sull'asse orizzontale. Le **fasce arancioni** sono i burst trovati da Kleinberg; più la fascia è scura, più è alto il livello (1-3), cioè quanto il ritmo nel burst supera quello di fondo del periodo (circa 2×, 4×, 8× con s = 2). La scala verticale è diversa in ogni riga, perché i burst sono calcolati rispetto al fondo del proprio periodo: si confrontano i picchi **dentro** una riga, non tra righe.

- **`comments_bursts_volume.png` — volume.** Barre blu = commenti per settimana. Una fascia indica settimane con molti più commenti del normale per quel periodo: lì il dibattito si è acceso per quantità. Nel 2003-2016 e nel 2021-2024 quasi ogni barra ha la sua fascia: con pochi articoli commentati il fondo è vicino a zero e ogni settimana con commenti risulta eccezionale, quindi quei due periodi dicono poco.
- **`comments_bursts_chain.png` — catene.** Ogni punto è un giorno con commenti: quota di commenti che sono risposte a una risposta (catena ≥ 3). Una fascia indica giorni in cui la discussione si allunga più del solito, cioè si risponde alle risposte invece di commentare l'articolo. Il pannello 2003-2016 è vuoto e il 2017-2020 parte da aprile 2018, perché prima il NYT non registrava risposte a risposte.
- **`comments_bursts_exchange.png` — botta e risposta.** Come il precedente, ma la quota è quella delle risposte dentro uno scambio reciproco tra due utenti (A risponde a B e B risponde ad A). Una fascia indica giorni di confronto diretto e ripetuto tra persone.

Nei due grafici del conflitto i punti isolati a 0 o a 1 sono quasi sempre giorni con pochissimi commenti (per esempio 2 commenti, 2 in catena = 1,0): pesano poco nel modello e da soli non generano un burst. Per questo una fascia può coprire punti non altissimi ma su giorni con molti commenti, mentre un punto a 1,0 può restare fuori. Per sapere se un burst di conflitto è solo l'effetto di un articolo molto commentato, va confrontato con il grafico del volume: è quello che fa §4.4.

In [ ]:
def plot_bursts(name: str, ylabel: str, filename: str) -> None:
    fig, axes = plt.subplots(len(period_labels), 1, figsize=(11, 2.3 * len(period_labels)))
    for ax, period in zip(axes, period_labels):
        series, states = series_store[(period, name)]
        if series.empty:
            ax.text(0.5, 0.5, "nessun dato (prima dell'inizio delle catene)", transform=ax.transAxes, ha="center", va="center", color="#777")
            ax.set_title(period, loc="left", fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])
            continue
        values = series["totali"] if SERIES[name]["relevant"] is None else series["rilevanti"] / series["totali"]
        width = pd.Timedelta(days=7 if SERIES[name]["unit"] == "W" else 1)
        for start, end, level in state_runs(states, series.index):
            ax.axvspan(series.index[start], series.index[end] + width, color=PERIOD_COLORS[1], alpha=0.12 + 0.08 * level, linewidth=0)
        if SERIES[name]["relevant"] is None:
            ax.bar(series.index, values, width=width * 0.85, color=MAIN_COLOR, align="edge")
        else:
            ax.plot(series.index, values, "o", markersize=2.5, color=MAIN_COLOR)
        ax.set_title(period, loc="left", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=8)
    fig.suptitle(f"Burst di Kleinberg — {name} (fasce arancioni, più scure = livello più alto)", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()


plot_bursts("volume", "commenti / settimana", "comments_bursts_volume.png")
plot_bursts("catena", "quota in catena / giorno", "comments_bursts_chain.png")
plot_bursts("botta_e_risposta", "quota botta e risposta / giorno", "comments_bursts_exchange.png")

### 4.4 Burst di conflitto senza burst di volume

Un burst di conflitto che cade dentro un burst di volume può essere solo l'effetto di un articolo molto commentato. I più interessanti sono quelli che **non** si sovrappongono a nessun burst di volume: pochi commenti, ma molto litigiosi. `in_burst_volume` = il burst di conflitto ha almeno un giorno dentro un burst di volume dello stesso periodo.

**Livello** (colonna `livello` nelle tabelle sotto): stato più alto raggiunto dall'automa di Kleinberg dentro il burst. Ogni stato corrisponde a un ritmo atteso multiplo del fondo del periodo (s = 2):

| Stato | Volume (Poisson) | Conflitto (binomiale) |
|---|---|---|
| 0 | λ₀ = media settimanale dei commenti nel periodo | p₀ = quota del periodo (es. catena 2025+: 0,113) |
| 1 | 2 · λ₀ | 2 · p₀ = 0,226 |
| 2 | 4 · λ₀ | 4 · p₀ = 0,452 |
| 3 | 8 · λ₀ | 8 · p₀ = 0,904 |

*Come viene assegnato.* Per ogni unità di tempo Viterbi sceglie lo stato che spiega meglio il dato, bilanciando due costi: quanto il conteggio (o la quota) osservato è verosimile sotto quello stato, e il costo di salita `γ · ln(T)` per ogni livello in più (circa 6 sulla serie giornaliera del 2025-presente); scendere è gratis. Per questo un singolo giorno con pochi commenti non fa salire di livello: il guadagno di verosimiglianza non copre il costo. Ignorando il costo di salita, le soglie oltre cui conviene il livello successivo sono:

- volume: livello 1 sopra circa 1,4 × λ₀, livello 2 sopra circa 2,9 ×, livello 3 sopra circa 5,8 ×;
- catena 2025-presente: livello 1 sopra circa il 16% di risposte in catena, livello 2 sopra il 33%, livello 3 sopra il 72% (purché i commenti del giorno siano abbastanza da superare il costo di salita).

*Cosa tenere presente.*

- Il livello è **relativo al periodo**: un livello 3 del 2021-2024 non equivale a un livello 3 del 2025-presente, perché λ₀ e p₀ sono diversi.
- Per il volume il livello è **saturo**: tutti e quattro i periodi arrivano a 3, perché i picchi settimanali superano spesso di molto 5,8 × il fondo e oltre il livello 3 non si distingue più. Per ordinare i burst di volume conta di più `commenti` (secondo criterio di ordinamento nelle tabelle).
- Nei burst di conflitto il livello è quasi sempre 1: i livelli 2-3 richiedono quote molto alte su giorni con molti commenti, quindi sono rari e sono i segnali più forti (es. 2020-02-11, livello 3 in botta e risposta).

In [ ]:
def volume_burst_days(period: str) -> set:
    series, states = series_store[(period, "volume")]
    days = set()
    for start, end, _ in state_runs(states, series.index):
        days |= set(pd.date_range(series.index[start], series.index[end] + pd.Timedelta(days=6), freq="D"))
    return days


volume_days = {p: volume_burst_days(p) for p in period_labels}
conflict = bursts[bursts["serie"] != "volume"].copy()
conflict["in_burst_volume"] = [
    bool(set(pd.date_range(row.inizio, row.fine, freq="D")) & volume_days[row.periodo]) for row in conflict.itertuples()
]
bursts = bursts.merge(conflict[["serie", "periodo", "inizio", "in_burst_volume"]], on=["serie", "periodo", "inizio"], how="left")

display(conflict.groupby(["serie", "in_burst_volume"]).size().unstack(fill_value=0))
conflict[~conflict["in_burst_volume"]].sort_values(["livello", "commenti"], ascending=False).head(10)[
    ["serie", "periodo", "inizio", "fine", "livello", "commenti", "quota_nel_burst", "quota_periodo", "articoli_principali"]
]

In [ ]:
# Burst di livello piu' alto per serie: le date e gli articoli da leggere per primi
for name in SERIES:
    print(f"\n=== {name}")
    top = bursts[bursts["serie"] == name].sort_values(["livello", "commenti"], ascending=False).head(8)
    display(top[["periodo", "inizio", "fine", "livello", "commenti", "giorno_di_picco", "articoli_principali"]])

### 4.5 Sensibilità ai parametri

Griglia `s ∈ {1.5, 2, 3}` × `γ ∈ {0.5, 1, 2}` (riferimento: s = 2, γ = 1, i valori di Kleinberg). Per ogni serie, sommando i 4 periodi: numero di burst, unità di tempo in burst e Jaccard tra le unità in burst e quelle del riferimento. Il volume è riportato anche con unità giornaliera, per documentare la scelta della settimana.

Poi, per ogni burst del riferimento, `stabilita` = quota delle 9 configurazioni in cui almeno metà delle sue unità resta in un burst. I burst con stabilità bassa dipendono dai parametri e vanno citati con cautela.

In [ ]:
S_GRID = [1.5, 2.0, 3.0]
GAMMA_GRID = [0.5, 1.0, 2.0]


def burst_units(states: np.ndarray, index: pd.Index) -> set:
    return set(index[states > 0])


sensitivity_rows = []
per_config_units = {}
daily_volume_lengths = []
for name in list(SERIES) + ["volume_giorno"]:
    for s in S_GRID:
        for gamma in GAMMA_GRID:
            units, n_bursts = set(), 0
            for period in period_labels:
                if name == "volume_giorno":
                    df_period = comments[comments["period"] == period]
                    counts = df_period.groupby("day").size()
                    counts = counts.reindex(pd.date_range(counts.index.min(), counts.index.max(), freq="D"), fill_value=0)
                    states = kleinberg_counts(counts.values, s=s, gamma=gamma)
                    index = counts.index
                else:
                    series, _ = series_store[(period, name)]
                    states = detect(series, name, s=s, gamma=gamma)
                    index = series.index
                units |= {(period, u) for u in burst_units(states, index)}
                n_bursts += len(state_runs(states, index))
                if (name, s, gamma) == ("volume_giorno", KLEINBERG_S, KLEINBERG_GAMMA):
                    daily_volume_lengths += [end - start + 1 for start, end, _ in state_runs(states, index)]
            per_config_units[(name, s, gamma)] = units
            sensitivity_rows.append({"serie": name, "s": s, "gamma": gamma, "burst": n_bursts, "unita_in_burst": len(units)})

sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity["jaccard_vs_riferimento"] = [
    round(len(per_config_units[(r.serie, r.s, r.gamma)] & per_config_units[(r.serie, KLEINBERG_S, KLEINBERG_GAMMA)])
          / max(1, len(per_config_units[(r.serie, r.s, r.gamma)] | per_config_units[(r.serie, KLEINBERG_S, KLEINBERG_GAMMA)])), 2)
    for r in sensitivity.itertuples()
]
lengths = pd.Series(daily_volume_lengths)
print(f"Volume giornaliero (riferimento): {len(lengths)} burst, durata mediana {lengths.median():.0f} giorni, "
      f"{(lengths <= 2).mean():.0%} di 1-2 giorni")
sensitivity.pivot_table(index=["serie", "s"], columns="gamma", values=["burst", "jaccard_vs_riferimento"])

**Come leggere la tabella sopra.** Una riga per serie e valore di `s`, una colonna per valore di `γ`, due blocchi:

- **`burst`**: numero di burst trovati con quella configurazione, sommando i 4 periodi;
- **`jaccard_vs_riferimento`**: quanto i giorni (o le settimane) in burst coincidono con quelli del riferimento s = 2, γ = 1. 1 = stessi giorni/settimane, 0 = nessuno in comune; la cella del riferimento vale 1 per costruzione.

Per il **volume** (settimanale) i Jaccard restano sopra 0,85 ovunque: i burst non dipendono dai parametri, e `γ` cambia quasi nulla (con conteggi settimanali così grandi la verosimiglianza domina il costo di salita). Per **catena** e **botta e risposta** i Jaccard scendono a 0,3-0,65 quando `s` passa a 1,5 o 3: i burst di conflitto dipendono soprattutto da `s`, molto meno da `γ`. Con `s` più basso i livelli sono più vicini al fondo e i burst aumentano; con `s` più alto ne restano solo i più forti.

La riga **`volume_giorno`** e la riga stampata sopra la tabella servono a giustificare la settimana: con l'unità giornaliera escono più del doppio dei burst, quasi tutti di 1-2 giorni, cioè uno per ogni giorno con un articolo molto commentato.

In [ ]:
def burst_stability(row) -> float:
    series, _ = series_store[(row.periodo, row.serie)]
    start = pd.Timestamp(row.inizio)
    if SERIES[row.serie]["unit"] == "W":
        start = start.to_period("W-SUN").start_time
    units = {(row.periodo, u) for u in series.index if start <= u <= pd.Timestamp(row.fine)}
    kept = [
        len(units & per_config_units[(row.serie, s, gamma)]) >= len(units) / 2
        for s in S_GRID for gamma in GAMMA_GRID
    ]
    return round(float(np.mean(kept)), 2)


bursts["stabilita"] = [burst_stability(row) for row in bursts.itertuples()]
bursts.to_csv(OUT_BURSTS, index=False)
log.info("Scritto %s (%d burst)", OUT_BURSTS.relative_to(BASE_DIR), len(bursts))

bursts.groupby("serie")["stabilita"].describe()[["count", "mean", "min", "50%"]].assign(
    stabili_almeno_078=bursts.groupby("serie")["stabilita"].apply(lambda s: int((s >= 7 / 9).sum()))
)

**Come leggere la tabella sopra.** Una riga per serie; le colonne descrivono la `stabilita` dei burst trovati con la configurazione di riferimento (s = 2, γ = 1). `stabilita` di un burst = quota delle 9 configurazioni della griglia in cui almeno metà dei suoi giorni (o settimane) resta in un burst: 1 = c'è sempre, 0,44 = c'è in 4 configurazioni su 9.

- **`count`**: numero di burst della serie;
- **`mean`, `min`, `50%`**: stabilità media, minima e mediana;
- **`stabili_almeno_078`**: burst presenti in almeno 7 configurazioni su 9, cioè quelli da usare con fiducia nelle conclusioni.

Il **volume** è quasi tutto stabile (mediana 1, 85 burst su 96). Per **catena** e **botta e risposta** la mediana è 0,67 (6 configurazioni su 9) e solo 16 burst per serie superano la soglia: gli altri compaiono o spariscono a seconda di `s` e vanno citati con cautela. La stabilità di ogni burst è nella colonna `stabilita` di `comments_bursts.csv`.

## 5. NYT Picks contro i commenti più apprezzati dai lettori

I *NYT Picks* (`editors_selection`) sono i commenti che la redazione mette in evidenza. Le raccomandazioni (`recommendations`) sono la scelta dei lettori. Domanda: **la redazione premia gli stessi commenti che premiano i lettori, e soprattutto parlano degli stessi soggetti?**

Percorso:

1. quanti sono i Picks e dove stanno;
2. dove si collocano i Picks nella classifica per raccomandazioni del loro articolo;
3. gruppo di controllo appaiato per articolo: i non-Picks con più raccomandazioni;
4. NER (PERSON / ORG / GPE) sui due gruppi;
5. entità che distinguono i due gruppi, con il metodo di Monroe et al. (2008).

### 5.1 Quanti sono e dove stanno

In [ ]:
picks = comments[comments["editors_selection"]]
log.info("NYT Picks: %d su %d commenti (%.2f%%), su %d articoli",
         len(picks), len(comments), len(picks) / len(comments) * 100, picks["article_id"].nunique())

articles_with_picks = picks["article_id"].unique()
in_pick_articles = comments[comments["article_id"].isin(articles_with_picks)]

picks_by_year = comments.groupby("year").agg(
    commenti=("comment_id", "size"),
    picks=("editors_selection", "sum"),
    articoli_con_commenti=("article_id", "nunique"),
)
picks_by_year["articoli_con_picks"] = picks.groupby("year")["article_id"].nunique().reindex(picks_by_year.index, fill_value=0)
picks_by_year["picks_pct"] = (picks_by_year["picks"] / picks_by_year["commenti"] * 100).round(2)
display(picks_by_year[picks_by_year["commenti"] > 0])

pd.Series({
    "picks": len(picks),
    "di cui risposte": int(picks["is_reply"].sum()),
    "% sul totale dei commenti": round(len(picks) / len(comments) * 100, 2),
    "articoli con picks": len(articles_with_picks),
    "picks per articolo (mediana)": picks.groupby("article_id").size().median(),
    "% picks sui commenti degli articoli con picks": round(len(picks) / len(in_pick_articles) * 100, 2),
})

I Picks sono concentrati nel 2017-2020 e quasi assenti dal 2022 (23 nel 2025-2026, nessuno nel 2022-2024): il confronto di questa sezione descrive soprattutto la scelta editoriale dell'era Trump I. Il 2025-presente è incluso ma pesa poco.

### 5.2 Picks e gradimento dei lettori

Per ogni Pick: posizione del commento nella classifica per raccomandazioni del suo articolo (1 = il più raccomandato). Se la redazione scegliesse i commenti più apprezzati, i Picks occuperebbero le prime posizioni.

In [ ]:
ranked = in_pick_articles.copy()
ranked["rank_raccomandazioni"] = ranked.groupby("article_id")["recommendations"].rank(ascending=False, method="min")
ranked["n_picks_articolo"] = ranked.groupby("article_id")["editors_selection"].transform("sum")
pick_ranks = ranked[ranked["editors_selection"]]

pd.Series({
    "posizione mediana di un pick": pick_ranks["rank_raccomandazioni"].median(),
    "picks tra i primi 10 dell'articolo %": round((pick_ranks["rank_raccomandazioni"] <= 10).mean() * 100, 1),
    # quanti Picks si otterrebbero prendendo, in ogni articolo, i k commenti piu' raccomandati (k = numero di Picks)
    "picks che sarebbero scelti anche per raccomandazioni %": round(
        (pick_ranks["rank_raccomandazioni"] <= pick_ranks["n_picks_articolo"]).mean() * 100, 1
    ),
    "picks oltre la 30a posizione %": round((pick_ranks["rank_raccomandazioni"] > 30).mean() * 100, 1),
})

### 5.3 Gruppo di controllo appaiato per articolo

Per ogni articolo con Picks e per ogni tipo di commento (commento all'articolo / risposta) si prendono **tanti non-Picks quanti sono i Picks**, scegliendo quelli con più raccomandazioni (a parità, `comment_id` più basso, per riproducibilità).

**Perché appaiato.** Con una soglia globale di raccomandazioni i commenti più apprezzati verrebbero dal 2025-2026 (68% dei commenti, articoli molto più letti), mentre i Picks vengono dal 2017-2020: la NER misurerebbe la differenza tra periodi e temi, non tra scelta della redazione e scelta dei lettori. Appaiando per articolo i due gruppi commentano lo stesso pezzo, nello stesso momento, davanti allo stesso pubblico. Appaiare anche il tipo evita che un gruppo contenga più risposte (più brevi e più legate al thread) dell'altro.

In [ ]:
def matched_top_liked(df: pd.DataFrame) -> pd.DataFrame:
    """Per (articolo, tipo): i k non-Picks con piu' raccomandazioni, k = numero di Picks di quel (articolo, tipo)."""
    quota = df[df["editors_selection"]].groupby(["article_id", "is_reply"]).size().rename("k")
    candidates = (
        df[~df["editors_selection"]]
        .merge(quota, left_on=["article_id", "is_reply"], right_index=True)
        .sort_values(["article_id", "is_reply", "recommendations", "comment_id"], ascending=[True, True, False, True])
    )
    candidates["pos"] = candidates.groupby(["article_id", "is_reply"]).cumcount()
    return candidates[candidates["pos"] < candidates["k"]].drop(columns=["k", "pos"])


top_liked = matched_top_liked(in_pick_articles)
groups = pd.concat([picks.assign(gruppo="NYT Picks"), top_liked.assign(gruppo="Più raccomandati")], ignore_index=True)
log.info("Gruppi: %d NYT Picks, %d piu' raccomandati (stessi articoli e tipi)", len(picks), len(top_liked))

groups.groupby("gruppo").agg(
    commenti=("comment_id", "size"),
    risposte=("is_reply", "sum"),
    raccomandazioni_mediane=("recommendations", "median"),
    parole_mediane=("word_count", "median"),
    ore_mediane_dalla_pubblicazione=("hours_since_publication", "median"),
    risposte_ricevute_mediane=("reply_count", "median"),
    commenti_2025_presente=("period", lambda s: int((s == "2025-presente").sum())),
).T

### 5.4 NER sui due gruppi

Stessa impostazione del notebook 06: spaCy `en_core_web_sm` (solo il componente NER), etichette PERSON / ORG / GPE, conteggio = **commenti distinti** che citano l'entità (un commento che ripete "Trump" cinque volte conta una volta).

La NER gira su tutti i commenti dei 90 articoli con Picks, non solo sui due gruppi: quel corpus serve da sfondo per il metodo di Monroe (§5.5). Le entità grezze sono salvate in `comments_ner_raw.parquet`; le esecuzioni successive calcolano solo i commenti mancanti.

**Normalizzazione** (ripresa dal notebook 06, senza le regole pensate per i titoli e i dateline degli articoli):

- possessivi, articolo iniziale, iniziali intermedie ("Donald J. Trump" → "Donald Trump"), parole tutte maiuscole rese in Title Case (nei commenti "TRUMP" è frequente);
- alias di sigle e varianti (U.S. → United States, G.O.P./GOP → Republican Party, DHS → Department of Homeland Security…);
- cognome → nome completo quando una forma completa domina (≥ 60%) tra quelle con quel cognome;
- correzioni di etichetta note di `en_core_web_sm` (es. "Trump" etichettato ORG);
- escluse le entità tautologiche (ICE, NYT, United States), come nel notebook 06.

In [ ]:
import spacy

SPACY_MODEL = "en_core_web_sm"
ENTITY_LABELS = ["PERSON", "ORG", "GPE"]
OUT_NER_RAW = PROCESSED_DIR / "comments_ner_raw.parquet"


def run_ner(df: pd.DataFrame) -> pd.DataFrame:
    """Entita' grezze (comment_id, text, label) con cache su disco: calcola solo i commenti non ancora processati."""
    cached = pd.read_parquet(OUT_NER_RAW) if OUT_NER_RAW.exists() else pd.DataFrame(columns=["comment_id", "text", "label"])
    done = set(pd.read_parquet(OUT_NER_RAW.with_suffix(".done.parquet"))["comment_id"]) if OUT_NER_RAW.with_suffix(".done.parquet").exists() else set()
    todo = df[~df["comment_id"].isin(done)]
    if len(todo):
        log.info("NER su %d commenti (%d gia' in cache)...", len(todo), len(df) - len(todo))
        nlp = spacy.load(SPACY_MODEL, disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])
        rows = []
        for cid, doc in zip(todo["comment_id"], nlp.pipe(todo["body"].fillna(""), batch_size=256)):
            rows.extend((cid, ent.text, ent.label_) for ent in doc.ents if ent.label_ in ENTITY_LABELS)
        cached = pd.concat([cached, pd.DataFrame(rows, columns=["comment_id", "text", "label"])], ignore_index=True)
        cached.to_parquet(OUT_NER_RAW, index=False)
        pd.DataFrame({"comment_id": sorted(done | set(todo["comment_id"]))}).to_parquet(OUT_NER_RAW.with_suffix(".done.parquet"), index=False)
        log.info("Scritto %s", OUT_NER_RAW.relative_to(BASE_DIR))
    return cached[cached["comment_id"].isin(df["comment_id"])]


raw_ents = run_ner(in_pick_articles)
log.info("Entita' grezze sui commenti degli articoli con Picks: %d", len(raw_ents))

In [ ]:
ALIASES = {
    "U.S.": "United States", "U.S": "United States", "US": "United States", "USA": "United States",
    "U.S.A.": "United States", "America": "United States", "the United States": "United States",
    "N.Y.C.": "New York City", "NYC": "New York City", "L.A.": "Los Angeles", "D.C.": "Washington, D.C.",
    "Immigration and Customs Enforcement": "ICE", "I.C.E.": "ICE", "U.S. Immigration and Customs Enforcement": "ICE",
    "Department of Homeland Security": "Department of Homeland Security", "Homeland Security": "Department of Homeland Security",
    "D.H.S.": "Department of Homeland Security", "DHS": "Department of Homeland Security",
    "Justice Department": "Department of Justice", "DOJ": "Department of Justice",
    "Customs and Border Protection": "Customs and Border Protection", "C.B.P.": "Customs and Border Protection", "CBP": "Customs and Border Protection",
    "G.O.P.": "Republican Party", "GOP": "Republican Party", "Republicans": "Republican Party", "Democrats": "Democratic Party",
    "New York Times": "The New York Times", "Times": "The New York Times", "NYT": "The New York Times",
    "F.B.I.": "FBI", "A.C.L.U.": "ACLU", "American Civil Liberties Union": "ACLU",
    "Trump Administration": "Trump administration", "Obama Administration": "Obama administration",
    "Biden Administration": "Biden administration",
    "Tom Homan": "Thomas Homan", "Barack Obama": "Barack Obama", "Obama": "Barack Obama",
    "NY Times": "The New York Times", "NYTimes": "The New York Times", "the Times": "The New York Times",
    "Republican party": "Republican Party", "Democratic party": "Democratic Party", "Dems": "Democratic Party",
    "Melania": "Melania Trump",
}
# Nei commenti le maiuscole sono irregolari ("congress", "republican party"): alias cercati senza distinzione di maiuscole
ALIASES_CI = {k.lower(): v for k, v in ALIASES.items()}
CANONICAL_CASE = {"congress": "Congress", "senate": "Senate", "house": "House", "white house": "White House",
                  "supreme court": "Supreme Court", "border patrol": "Border Patrol"}
RELABEL = {
    ("Trump", "ORG"): "PERSON", ("Trump", "GPE"): "PERSON", ("Obama", "ORG"): "PERSON", ("Biden", "ORG"): "PERSON",
    ("Kirstjen Nielsen", "ORG"): "PERSON", ("ACLU", "PERSON"): "ORG", ("Department of Homeland Security", "GPE"): "ORG",
    ("Border Patrol", "PERSON"): "ORG", ("Social Security", "ORG"): None,
    ("Melania Trump", "GPE"): "PERSON", ("Nielsen", "GPE"): "PERSON", ("Twitter", "PERSON"): "ORG",
    ("Barack Obama", "GPE"): "PERSON", ("Donald Trump", "ORG"): "PERSON", ("Donald Trump", "GPE"): "PERSON",
}
ET_AL_RE = re.compile(r"\s+et al\.?$", flags=re.IGNORECASE)
EXCLUDE = {"ORG": {"ICE", "The New York Times"}, "GPE": {"United States"}, "PERSON": set()}
MAX_ENTITY_TOKENS = 5
NOISE = {"al.", "al", "et al."}  # resti di "et al." spezzati dal tokenizer
SURNAME_DOMINANCE = 0.6

POSSESSIVE_RE = re.compile(r"[’']s?$")
LEADING_THE_RE = re.compile(r"^the\s+", flags=re.IGNORECASE)
MIDDLE_INITIAL_RE = re.compile(r"(?<=\w)\s+[A-Z]\.(?=\s+\w)")


def normalize(text: str, label: str) -> tuple[str, str] | None:
    t = POSSESSIVE_RE.sub("", text.strip())
    t = ET_AL_RE.sub("", LEADING_THE_RE.sub("", t)).strip(" ,;:\"“”()")
    if len(t) < 2 or len(t.split()) > MAX_ENTITY_TOKENS or t.lower() in NOISE:
        return None
    if t.isupper() and len(t) > 4 and "." not in t:
        t = t.title()
    t = ALIASES.get(t, ALIASES_CI.get(t.lower(), CANONICAL_CASE.get(t.lower(), t)))
    if label == "PERSON":
        t = MIDDLE_INITIAL_RE.sub("", t)
    if (t, label) in RELABEL:
        label = RELABEL[(t, label)]
        if label is None:
            return None
    return t, label


def normalize_entities(raw: pd.DataFrame) -> pd.DataFrame:
    """(comment_id, entity, label) normalizzate, una riga per commento-entita'."""
    norm = [(cid, *n) for cid, text, label in raw.itertuples(index=False) if (n := normalize(text, label))]
    ents = pd.DataFrame(norm, columns=["comment_id", "entity", "label"])

    full_names = ents[(ents["label"] == "PERSON") & ents["entity"].str.contains(" ")]["entity"].value_counts()
    surname_map = {}
    for surname, variants in full_names.groupby(full_names.index.str.split().str[-1]):
        if variants.max() / variants.sum() >= SURNAME_DOMINANCE:
            surname_map[surname] = variants.idxmax()
    single = (ents["label"] == "PERSON") & ~ents["entity"].str.contains(" ")
    ents.loc[single, "entity"] = ents.loc[single, "entity"].map(lambda s: surname_map.get(s, s))

    ents = ents[~ents.apply(lambda r: r["entity"] in EXCLUDE[r["label"]], axis=1)]
    return ents.drop_duplicates(["comment_id", "entity", "label"])


entities = normalize_entities(raw_ents)
print("Cognome -> nome completo:", {s: entities[entities["entity"].str.endswith(s)]["entity"].value_counts().index[0]
                                    for s in ["Trump", "Obama", "Nielsen", "Sessions", "Miller"]
                                    if entities["entity"].str.endswith(s).any()})

group_entities = entities.merge(groups[["comment_id", "gruppo"]], on="comment_id")
group_sizes = groups["gruppo"].value_counts()

TOP_ENTITIES = 12
top_by_group = (
    group_entities.groupby(["gruppo", "label", "entity"]).size().rename("commenti").reset_index()
    .assign(pct=lambda x: (x["commenti"] / x["gruppo"].map(group_sizes) * 100).round(1))
    .sort_values(["gruppo", "label", "commenti", "entity"], ascending=[True, True, False, True])
    .groupby(["gruppo", "label"]).head(TOP_ENTITIES)
)
top_by_group["rank"] = top_by_group.groupby(["gruppo", "label"]).cumcount() + 1
for label in ENTITY_LABELS:
    print(f"\n=== {label}: top {TOP_ENTITIES} per gruppo (% commenti del gruppo)")
    display(
        top_by_group[top_by_group["label"] == label]
        .assign(cella=lambda x: x["entity"] + " (" + x["pct"].astype(str) + "%)")
        .pivot(index="rank", columns="gruppo", values="cella")
    )

### 5.5 Entità che distinguono i due gruppi: metodo di Monroe et al. (2008)

**Il metodo.** Monroe, Colaresi & Quinn (2008), *Fightin' Words: Lexical Feature Selection and Evaluation for Identifying the Content of Political Conflict*. Per ogni entità *w* si calcola il **log-odds ratio** tra i due gruppi, con un **prior di Dirichlet informativo** stimato su un corpus di sfondo, e lo si divide per la sua deviazione standard:

$$\delta_w = \log\frac{y_{w}^{P} + \alpha_w}{n^{P} + \alpha_0 - y_{w}^{P} - \alpha_w} - \log\frac{y_{w}^{R} + \alpha_w}{n^{R} + \alpha_0 - y_{w}^{R} - \alpha_w}, \qquad z_w = \frac{\delta_w}{\sqrt{\frac{1}{y_{w}^{P} + \alpha_w} + \frac{1}{y_{w}^{R} + \alpha_w}}}$$

- *y_w^P*, *y_w^R*: commenti che citano *w* tra i Picks (P) e tra i più raccomandati (R); *n^P*, *n^R*: totale delle coppie commento-entità del gruppo;
- *α_w* = frequenza di *w* nel corpus di sfondo (tutti i 52.732 commenti dei 90 articoli con Picks) riscalata in modo che il peso totale del prior *α₀* sia pari al numero di coppie commento-entità dei due gruppi insieme (prior e dati pesano uguale);
- *z_w* > 0: entità tipica dei Picks; *z_w* < 0: tipica dei più raccomandati; |*z*| > 1,96 ≈ differenza significativa al 5% (senza correzione per confronti multipli: va letta come graduatoria, non come test formale).

**Perché questo metodo e non la differenza di percentuali.** Con circa 800 commenti per gruppo molte entità compaiono in 1-5 commenti.

- La **differenza di percentuali** premia le entità frequenti (una differenza di 3 punti su Trump, citato nel 30% dei commenti, è rumore; su un nome citato nel 2% è enorme) e non dice se la differenza è più grande del caso.
- Il **log-odds semplice** ha il problema opposto: un'entità citata 3 volte in un gruppo e 0 nell'altro ottiene un rapporto infinito, e le entità rare dominano la graduatoria.
- Monroe risolve entrambi: il **prior** riporta le entità rare verso la loro frequenza "normale" nel corpus di sfondo (servono più osservazioni per allontanarsene), e la **divisione per la deviazione standard** tiene conto di quanto è incerta la stima. Il risultato è una graduatoria che premia le entità **sia frequenti sia sbilanciate**. È il metodo standard in linguistica computazionale e scienza politica per "quali parole distinguono due gruppi di testi" (discorsi di partiti diversi, recensioni positive e negative) e qui si applica alle entità al posto delle parole.
- Il corpus di sfondo sono i commenti degli **stessi 90 articoli**: il prior rappresenta di cosa parlano normalmente i lettori di quei pezzi, così il confronto non premia entità che sono solo tipiche dell'argomento degli articoli.

Si considerano solo le entità citate in almeno `MIN_GROUP_COMMENTS = 5` commenti dei due gruppi insieme.

In [ ]:
MIN_GROUP_COMMENTS = 5
Z_THRESHOLD = 1.96


def monroe_log_odds(counts_a: pd.Series, counts_b: pd.Series, background: pd.Series, prior_total: float) -> pd.DataFrame:
    """Log-odds ratio con prior di Dirichlet informativo (Monroe et al. 2008). Positivo = tipico del gruppo a."""
    vocab = counts_a.index.union(counts_b.index)
    y_a = counts_a.reindex(vocab, fill_value=0).astype(float)
    y_b = counts_b.reindex(vocab, fill_value=0).astype(float)
    alpha = background.reindex(vocab, fill_value=0).astype(float)
    # le entita' dei gruppi sono un sottoinsieme dei commenti di sfondo, quindi alpha > 0; il clip e' solo una garanzia
    alpha = (alpha / background.sum() * prior_total).clip(lower=1e-3)
    alpha_0 = prior_total
    n_a, n_b = y_a.sum(), y_b.sum()
    delta = (
        np.log((y_a + alpha) / (n_a + alpha_0 - y_a - alpha))
        - np.log((y_b + alpha) / (n_b + alpha_0 - y_b - alpha))
    )
    variance = 1 / (y_a + alpha) + 1 / (y_b + alpha)
    return pd.DataFrame({"y_a": y_a.astype(int), "y_b": y_b.astype(int), "delta": delta, "z": delta / np.sqrt(variance)})


key = group_entities["entity"] + " | " + group_entities["label"]
counts_picks = key[group_entities["gruppo"] == "NYT Picks"].value_counts()
counts_liked = key[group_entities["gruppo"] == "Più raccomandati"].value_counts()
background = (entities["entity"] + " | " + entities["label"]).value_counts()

log_odds = monroe_log_odds(counts_picks, counts_liked, background, prior_total=float(len(group_entities)))
log_odds = log_odds[(log_odds["y_a"] + log_odds["y_b"]) >= MIN_GROUP_COMMENTS].copy()
log_odds[["entity", "label"]] = log_odds.index.to_series().str.split(" | ", regex=False, expand=True).values
log_odds = log_odds.rename(columns={"y_a": "commenti_picks", "y_b": "commenti_piu_raccomandati"})
log_odds["pct_picks"] = (log_odds["commenti_picks"] / group_sizes["NYT Picks"] * 100).round(1)
log_odds["pct_piu_raccomandati"] = (log_odds["commenti_piu_raccomandati"] / group_sizes["Più raccomandati"] * 100).round(1)
log_odds["commenti_sfondo"] = background.reindex(log_odds.index).values
log_odds = log_odds.sort_values("z", ascending=False).reset_index(drop=True)[
    ["entity", "label", "commenti_picks", "pct_picks", "commenti_piu_raccomandati", "pct_piu_raccomandati", "commenti_sfondo", "delta", "z"]
]
log_odds[["delta", "z"]] = log_odds[["delta", "z"]].round(3)
log_odds.to_csv(OUT_PICKS_ENTITIES, index=False)
log.info("Scritto %s (%d entita' con almeno %d commenti)", OUT_PICKS_ENTITIES.relative_to(BASE_DIR), len(log_odds), MIN_GROUP_COMMENTS)

print(f"Entita' con |z| > {Z_THRESHOLD}: {(log_odds['z'].abs() > Z_THRESHOLD).sum()} su {len(log_odds)}")
display(log_odds.head(15))
log_odds.tail(15).iloc[::-1]

**Sensibilità al peso del prior.** Il peso totale del prior *α₀* è una scelta: più è grande, più le entità rare vengono riportate verso la frequenza di sfondo e meno z-score superano la soglia. Monroe et al. usano direttamente i conteggi del corpus di sfondo (qui sarebbe circa 30 volte il peso dei dati); il riferimento di questa sezione è peso uguale ai dati (1×). La tabella mostra quante entità superano |z| > 1,96 con pesi da 0,1× a 30× e quali sono.

In [ ]:
prior_rows = []
for scale in [0.1, 0.3, 1.0, 3.0, 30.0]:
    lo = monroe_log_odds(counts_picks, counts_liked, background, prior_total=scale * len(group_entities))
    lo = lo[(lo["y_a"] + lo["y_b"]) >= MIN_GROUP_COMMENTS]
    sig = lo[lo["z"].abs() > Z_THRESHOLD].sort_values("z", ascending=False)
    prior_rows.append({
        "peso_prior": f"{scale}x",
        "entita_significative": len(sig),
        "tipiche_picks": ", ".join(sig.index[sig["z"] > 0].str.split(" | ", regex=False).str[0]),
        "tipiche_piu_raccomandati": ", ".join(sig.index[sig["z"] < 0].str.split(" | ", regex=False).str[0]),
    })
pd.DataFrame(prior_rows)

In [ ]:
TOP_DISTINCTIVE = 15
side = pd.concat([log_odds.head(TOP_DISTINCTIVE), log_odds.tail(TOP_DISTINCTIVE)]).drop_duplicates("entity").sort_values("z")
colors = [PERIOD_COLORS[0] if z > 0 else PERIOD_COLORS[1] for z in side["z"]]
alphas = [1.0 if abs(z) > Z_THRESHOLD else 0.4 for z in side["z"]]

fig, ax = plt.subplots(figsize=(8, 0.32 * len(side) + 1.5))
for y, (z, color, alpha) in enumerate(zip(side["z"], colors, alphas)):
    ax.barh(y, z, color=color, alpha=alpha, height=0.7)
ax.set_yticks(range(len(side)), [f"{e} ({l})" for e, l in zip(side["entity"], side["label"])], fontsize=8)
ax.axvline(0, color="#555", linewidth=0.8)
for x in (-Z_THRESHOLD, Z_THRESHOLD):
    ax.axvline(x, color=MUTED_COLOR, linewidth=0.8, linestyle=":")
ax.set_xlabel("z-score log-odds (Monroe et al. 2008)")
ax.set_title("Entità tipiche dei NYT Picks (blu, a destra) e dei commenti più raccomandati (arancione, a sinistra)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_picks_entities.png", dpi=200, bbox_inches="tight")
plt.show()

**Come leggere il grafico.** Ogni barra è un'entità; la lunghezza è lo z-score di Monroe. Barre **blu a destra**: entità citate più spesso nei NYT Picks; barre **arancioni a sinistra**: entità citate più spesso nei commenti più raccomandati dai lettori, negli stessi articoli. Le barre piene superano |z| = 1,96 (linee tratteggiate), quelle chiare no e vanno lette come tendenza. I conteggi e le percentuali di ogni entità sono nella tabella sopra e in `comments_picks_entities.csv`.

### 5.6 Cosa dicono i risultati

- **La redazione non sceglie i commenti più apprezzati.** Un Pick è in mediana il 16° commento per raccomandazioni del suo articolo; solo il 41% dei Picks sarebbe selezionato prendendo i commenti con più raccomandazioni.
- **I Picks si distinguono per la forma, non per i soggetti.** Rispetto ai più raccomandati degli stessi articoli sono più lunghi (90 parole contro 60 in mediana), arrivano più tardi (circa 7 ore dalla pubblicazione contro 3,5), hanno meno raccomandazioni (120 contro 149) ma ricevono più risposte (4 contro 2).
- **Le entità citate sono quasi le stesse.** Donald Trump compare nel 26,8% dei Picks e nel 26,9% dei più raccomandati; nessuna entità supera |z| = 1,96 con il metodo di Monroe, a nessun peso del prior tra 0,3× e 30× (a 0,1×, cioè quasi senza prior, solo Honduras, tipica dei più raccomandati). Le tendenze non significative vanno nella direzione di Picks più "istituzionali e di policy" (Congress, DACA, Customs and Border Protection, Kirstjen Nielsen) e di commenti apprezzati più centrati sugli attori politici di parte (Republican Party, Stephen Miller), ma con questi numeri non sono distinguibili dal caso.
- **Lettura:** la selezione editoriale non sposta *di chi* si parla; se sposta qualcosa, lo fa nel *come* (commenti più lunghi e argomentati, più capaci di aprire discussioni). Per verificarlo servono misure sul testo (tono, sentiment, argomentazione), non sulle entità.

Nota: "Alex" e "Glenn" sono persone degli articoli citate con il solo nome (il protagonista di un articolo; "Ms. Glenn", autrice di un editoriale a cui i lettori si rivolgono direttamente): la NER non può ricondurle a un nome completo.

## 6. Topic dei commenti (BERTopic)

Obiettivo: di cosa parlano i lettori, con quali parole, e come cambia tra periodi. Stessa pipeline BERTopic del notebook 04 (articoli NYT), adattata a testi brevi e conversazionali.

**Decisioni** (motivate nelle sottosezioni):

| Decisione | Scelta | Dove |
|---|---|---|
| Pulizia | menzioni `@utente`, nome dell'utente a cui si risponde in apertura e URL rimossi prima di tutto | 6.1 |
| Soglia di lunghezza | **≥ 10 parole** sul testo pulito | 6.2 |
| Embedding | `all-MiniLM-L6-v2`, come notebook 04 e 07 (da sostituire con `all-mpnet-base-v2` in tutti i notebook in un secondo momento) | 6.3 |
| Addestramento | campione **bilanciato per periodo**, poi `transform` su tutti i commenti | 6.4 |
| Parametri | UMAP come notebook 04; `min_topic_size` adattato alla scala; c-TF-IDF + KeyBERTInspired | 6.5 |
| Outlier | **lasciati** in questa prima esecuzione, misurati per periodo e lunghezza | 6.6 |
| Stabilità | controllo rapido su 4 seed (AMI) | 6.8 |

### 6.1 Pulizia del testo

Le risposte si aprono quasi sempre con il nome della persona a cui si risponde (`@Margot. So true.`). Lasciato nel testo, il nome:

- entra tra le parole caratterizzanti dei topic (un utente molto attivo diventerebbe una "parola" di un topic);
- gonfia il conteggio delle parole (`@Mrs. Kravitz Regrettably, yes.` = 4 parole, di cui 1 di contenuto);
- è un dato personale.

Regole: (1) ogni `@` seguito dal nome visualizzato di un utente noto viene rimosso fino alla fine del nome (si prova il nome più lungo fino a 5 parole, perché i nomi contengono spazi: `@Mike L`, `@Marge Keller`); se nessun nome corrisponde si rimuove la parola subito dopo la `@`; (2) nelle risposte si rimuove anche il nome dell'utente a cui si risponde quando apre il testo senza `@` (`Rhys Yes. This is…`); (3) URL rimossi; (4) spazi normalizzati.

I nomi visualizzati servono solo qui, in memoria: sono letti dal Parquet non pseudonimizzato e non finiscono in nessun output. `body_clean` e `word_count_clean` sono aggiunti a `comments`.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MAX_NAME_TOKENS = 5


def load_display_names() -> tuple[set, pd.Series]:
    """Nomi visualizzati (solo in memoria, per la pulizia) e nome dell'utente a cui risponde ogni commento."""
    raw = pd.read_parquet(IN_COMMENTS, columns=["comment_id", "user_display_name", "parent_user_display_name"])
    names = set(raw["user_display_name"].dropna()) | set(raw["parent_user_display_name"].dropna())
    return names, raw.set_index("comment_id")["parent_user_display_name"]


def strip_mentions(text: str, names: set) -> str:
    out, i = [], 0
    while (j := text.find("@", i)) >= 0:
        out.append(text[i:j])
        rest = text[j + 1:]
        tokens = re.split(r"(\s+)", rest)  # separatori conservati per ricostruire la lunghezza esatta
        cut = None
        for k in range(min(MAX_NAME_TOKENS, (len(tokens) + 1) // 2), 0, -1):
            candidate = "".join(tokens[:2 * k - 1])
            if candidate.rstrip(":,;.!?") in names:
                cut = len(candidate)
                break
        if cut is None:
            match = re.match(r"[\w.\-]+", rest)
            cut = match.end() if match else 0
        out.append(" ")
        i = j + 1 + cut
    out.append(text[i:])
    return "".join(out)


def clean_body(body: str, parent_name: str | None, names: set) -> str:
    text = strip_mentions(body, names) if "@" in body else body
    text = re.sub(r"\s+", " ", URL_RE.sub(" ", text)).strip()
    if isinstance(parent_name, str) and parent_name and text.startswith(parent_name):
        text = text[len(parent_name):]
    return text.strip(" :,;-–—")


display_names, parent_names = load_display_names()
comments["body_clean"] = [
    clean_body(body or "", parent_names.get(cid), display_names)
    for cid, body in zip(comments["comment_id"], comments["body"])
]
comments["word_count_clean"] = comments["body_clean"].str.split().str.len().fillna(0).astype(int)
del display_names, parent_names

changed = comments["body_clean"] != comments["body"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
log.info("Testi modificati dalla pulizia (menzioni, nomi, URL): %d (%.1f%%)", changed.sum(), changed.mean() * 100)
comments.loc[changed, ["body", "body_clean"]].sample(6, random_state=1).map(lambda s: s[:110])

### 6.2 Lunghezza dei commenti e soglia

Un commento troppo breve non ha un tema riconoscibile ("So true.", "Horrific and outrageous.", "One can hope."): l'embedding lo colloca vicino ad altri commenti brevi per tono, non per argomento, e il topic che ne esce raccoglie esclamazioni invece di temi.

Sotto: distribuzione delle parole dopo la pulizia, quota esclusa a diverse soglie e campioni per fascia di lunghezza.

In [ ]:
wc = comments["word_count_clean"]
display(wc.describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(0).to_frame("parole (testo pulito)").T)

threshold_rows = []
for t in [5, 8, 10, 12, 15, 20]:
    below = wc < t
    threshold_rows.append({
        "soglia": t,
        "esclusi_pct": round(below.mean() * 100, 1),
        "esclusi_commenti_pct": round(below[~comments["is_reply"]].mean() * 100, 1),
        "esclusi_risposte_pct": round(below[comments["is_reply"]].mean() * 100, 1),
        **{f"esclusi_{p}_pct": round(below[comments["period"] == p].mean() * 100, 1) for p in period_labels},
    })
display(pd.DataFrame(threshold_rows).set_index("soglia"))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(wc.clip(upper=300), bins=np.arange(0, 302, 2), color=MAIN_COLOR)
for t, style in [(10, "-"), (12, ":")]:
    ax.axvline(t, color=PERIOD_COLORS[1], linewidth=1.2, linestyle=style, label=f"{t} parole")
ax.set_xlabel("parole per commento (testo pulito; > 300 accorpati a 300)")
ax.set_ylabel("commenti")
ax.set_title("Lunghezza dei commenti", loc="left")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_length_hist.png", dpi=200, bbox_inches="tight")
plt.show()

for lo, hi in [(5, 8), (8, 10), (10, 12), (12, 15)]:
    sample = comments.loc[(wc >= lo) & (wc < hi), "body_clean"].sample(6, random_state=3)
    print(f"\n--- {lo}-{hi - 1} parole")
    for text in sample:
        print("  ", text[:130])

**Scelta: `MIN_WORDS = 10`** (nell'intervallo 10-12 deciso a monte). Esclude il 6,1% dei commenti: 3,7% dei commenti all'articolo e 8,7% delle risposte, che sono più brevi; la quota esclusa è simile in tutti i periodi (4-7%). Nei campioni sotto le 8 parole prevalgono le reazioni ("Hurting people is the point.", "Thank you for your service and advice."); tra 8 e 11 parole i commenti sono misti, alcuni già con un tema riconoscibile ("Canada has strict immigration laws.", "Funding for ICE but nothing else."). Salire a 12 escluderebbe altri 2,6 punti (8,7% in tutto).

**Verifica a posteriori** (§6.6): se la fascia 10-11 parole finisse quasi tutta tra gli outlier, la soglia andrebbe alzata a 12. Risultato: outlier 61,6% nella fascia 10-11 contro 60,2% nella 12-14 e 52-57% oltre. Le due fasce più brevi si comportano allo stesso modo, quindi 10 è confermata.

In [ ]:
MIN_WORDS = 10
comments["topic_eligible"] = comments["word_count_clean"] >= MIN_WORDS
log.info("Commenti con almeno %d parole: %d su %d (%.1f%%)",
         MIN_WORDS, comments["topic_eligible"].sum(), len(comments), comments["topic_eligible"].mean() * 100)

### 6.3 Embedding

**Modello: `all-MiniLM-L6-v2`.** È addestrato su circa un miliardo di coppie di frasi, con una parte grande di testo conversazionale (Reddit, forum di domande e risposte): è adatto a commenti brevi e informali. È lo stesso modello dei notebook 04 (articoli NYT) e 07 (Guardian), quindi i topic dei commenti sono confrontabili con quelli degli articoli. Limite: circa 256 token (~190 parole), oltre il testo è troncato; riguarda il 3,9% dei commenti, accettato invece di spezzare il testo in blocchi come per il Guardian.

**Da rivedere:** in un secondo momento il modello sarà sostituito con `all-mpnet-base-v2` (più accurato, circa 5 volte più lento) in tutti i notebook insieme, per non rompere la confrontabilità.

**Costo e cache.** Gli embedding sono calcolati per **tutti** i commenti puliti (anche quelli sotto soglia, che costano poco), così cambiare `MIN_WORDS` non richiede di ricalcolarli. Sono salvati a blocchi di 20.000 in `data/processed/nyt/comments_embeddings_minilm/` (embedding + `comment_id` per blocco): se il calcolo si interrompe, riparte dal primo blocco mancante. La cartella ha nel nome la versione della pulizia (`CLEANING_VERSION`): se le regole di 6.1 cambiano, va alzata e gli embedding vengono ricalcolati.

Facendo embedding andiamo a trasformare ogni commento pulito in un vettore da 384 coordinate, poi BERTopic raggruppa i commenti in base a questi vettori: commenti con vettori vicini parlano di cose simili.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
CLEANING_VERSION = 1
EMBEDDING_CHUNK = 20_000
EMBEDDINGS_DIR = PROCESSED_DIR / f"comments_embeddings_minilm_v{CLEANING_VERSION}"


def compute_embeddings(df: pd.DataFrame) -> pd.DataFrame:
    """Embedding di body_clean per tutti i commenti, a blocchi con checkpoint. Ritorna DataFrame indicizzato per comment_id."""
    EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)
    ordered = df.sort_values("comment_id")[["comment_id", "body_clean"]].reset_index(drop=True)
    model = None
    for start in range(0, len(ordered), EMBEDDING_CHUNK):
        path = EMBEDDINGS_DIR / f"chunk_{start // EMBEDDING_CHUNK:03d}.npz"
        chunk = ordered.iloc[start:start + EMBEDDING_CHUNK]
        if path.exists() and np.array_equal(np.load(path)["comment_id"], chunk["comment_id"].values):
            continue
        model = model or SentenceTransformer(EMBEDDING_MODEL)
        log.info("Embedding blocco %d (%d commenti)...", start // EMBEDDING_CHUNK, len(chunk))
        vectors = model.encode(chunk["body_clean"].tolist(), batch_size=64, show_progress_bar=False)
        np.savez(path, comment_id=chunk["comment_id"].values, embedding=vectors.astype(np.float32))
    parts = [np.load(p) for p in sorted(EMBEDDINGS_DIR.glob("chunk_*.npz"))]
    ids = np.concatenate([p["comment_id"] for p in parts])
    vectors = np.concatenate([p["embedding"] for p in parts])
    return pd.DataFrame(vectors, index=pd.Index(ids, name="comment_id"))


all_embeddings = compute_embeddings(comments)
log.info("Embedding disponibili: %d x %d", *all_embeddings.shape)

### 6.4 Campione bilanciato per periodo

Il 68% dei commenti è del 2025-presente. Addestrando BERTopic su tutti, i temi degli altri periodi (pre 2025) finirebbero assorbiti nei temi del 2025-2026 o tra gli outlier, perché HDBSCAN forma un cluster solo dove i punti sono densi. Il modello si addestra quindi su un campione con **al massimo `N_PER_PERIOD` commenti per periodo** (tutti, se il periodo ne ha meno), estratti a caso con seed fisso; poi `transform` assegna un topic a tutti i commenti sopra soglia.

Senza bilanciamento il 2025+ (69%) dominerebbe. HDBSCAN forma cluster solo dove i punti sono densi, quindi i temi dei periodi piccoli finirebbero tra gli outlier o dentro i temi del 2025.

1. "Le dimensioni dei topic nel campione non riflettono il corpus". Il campione è deformato di proposito. Un topic tipico del 2003-2016 conta quasi 5 volte più di quanto peserebbe nel corpus (19,7% contro 4,0%). Uno tipico del 2025+ conta molto meno (27,4% contro 68,6%). Quindi i conteggi che BERTopic restituisce dopo fit (tipo "topic 3 = 2.100 commenti") non dicono quanto un tema sia diffuso tra i lettori.

2. "Le quote per periodo vanno lette dopo il transform (§6.7)". Il modello addestrato sul campione assegna poi un topic a tutti i 268.866 commenti, tramite transform. Solo quei conteggi rispecchiano il corpus reale. Tabelle e grafici della §6.7 (topic × periodo) usano quindi l'assegnazione completa, non quella del campione.

3. "Il 2003-2016 ha meno commenti della quota e resta sottorappresentato". La quota è 15.000, ma il 2003-2016 ha solo 10.768 commenti utili: entrano tutti e restano comunque sotto la quota. Nel campione pesa 19,7% contro il ~27% di 2017-2020 e 2025+. È sovrarappresentato rispetto al corpus, ma è comunque il periodo più debole nell'addestramento. Un suo tema ha meno punti per formare un cluster denso, quindi ha più probabilità di finire tra gli outlier.

Imprecisione nel testo: anche il 2021-2024 è sotto la quota (13.907 < 15.000), ma la frase cita solo il 2003-2016. La differenza è piccola (25,4% contro 27,4%), ma va detto.

In [ ]:
N_PER_PERIOD = 15_000
SAMPLE_SEED = 42

eligible = comments[comments["topic_eligible"]].reset_index(drop=True)
train = pd.concat(
    [g.sample(min(len(g), N_PER_PERIOD), random_state=SAMPLE_SEED) for _, g in eligible.groupby("period", observed=True)],
    ignore_index=True,
)
display(pd.DataFrame({
    "sopra_soglia": eligible["period"].value_counts(),
    "campione": train["period"].value_counts(),
}).reindex(period_labels))

train_docs = train["body_clean"].tolist()
train_embeddings = all_embeddings.loc[train["comment_id"]].values
eligible_docs = eligible["body_clean"].tolist()
eligible_embeddings = all_embeddings.loc[eligible["comment_id"]].values

### 6.5 Modello

- **UMAP**: stessi parametri del notebook 04 (`n_neighbors=15`, `n_components=5`, `min_dist=0`, coseno, `random_state=42`).
- **`min_topic_size = 100`**: il valore del notebook 04 (15) era tarato su 2.093 articoli; su circa 56.000 commenti produrrebbe centinaia di micro-topic (un thread che ripete la stessa battuta diventa un topic). 100 è circa lo 0,2% del campione.
- **Parole caratterizzanti**: due rappresentazioni. *c-TF-IDF* (default di BERTopic, come nel notebook 04): parole frequenti nel topic e rare negli altri. *KeyBERTInspired*: riordina le parole candidate per vicinanza semantica al centro del topic; con testi brevi e rumorosi dà parole più leggibili.
- **Stopword**: inglesi standard + "ice" (come nel notebook 04: presente ovunque, renderebbe le etichette tutte uguali) + contrazioni spezzate dal tokenizer ("don", "doesn", "ll", "ve"…). `min_df = 5`, unigrammi e bigrammi.

Cosa fa il modello:
1. Riceve gli embedding calcolati dalla sezione 6.3 e riduce le dimensioni dei vettori da 384 a 5
2. Cerca le zone dense nello spazio a 5 dimensioni, ogni zona densa diventa un topic. I punti che non si trovano in nessuna di queste zone sono classificati come outlier (-1)
3. Per ogni topic sceglie le parole che meglio lo descrivono attraverso c-TF-IDF (parole frequenti per quel topic ma rare negli altri).
4. Riordina le parole candidate per vicinanza semantica al centro del topic e su testi rumorosi dà etichette più leggibili.

- **OUTLIER**:BERTopic configura HDBSCAN con min_cluster_size = min_topic_size = 100 e non imposta min_samples, in questo caso la libreria usa quindi min_samples = min_cluster_size (100). Quindi un punto entra in un cluster solo se è denso rispetto ai suoi 100 vicini. Per eliminare alcuni outlier potremmo quindi: passare a BERTopic un HDBSCAN esplicito con min_samples più bassso (10-25), in questo modo gli outlier calerebbero ma anche i topic cambierebbero e andrebbero ricontrollati, così come la stabilità. 
Altrimenti potremmo tenere il modello attuale e riassegnare gli outlier con reduce_outliers (come fatto in notebook 04). I topic restano gli stessi, si aggiungono solo assegnazioni di articoli che soo più deboli e andrebbero assegnate con dei flag

In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from umap import UMAP

MIN_TOPIC_SIZE = 100
TOPIC_SEED = 42
COMMENT_STOPWORDS = list(ENGLISH_STOP_WORDS | {
    "ice", "don", "doesn", "didn", "isn", "aren", "wasn", "weren", "won", "wouldn", "shouldn", "couldn",
    "ll", "ve", "re", "just",
})
logging.getLogger("BERTopic").setLevel(logging.WARNING)
# il caricamento del modello fa decine di richieste HEAD a HuggingFace, ognuna loggata a INFO
for noisy in ("httpx", "huggingface_hub", "sentence_transformers"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
embedding_model = SentenceTransformer(EMBEDDING_MODEL)


def make_topic_model(seed: int, with_keybert: bool = True) -> BERTopic:
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=seed),
        vectorizer_model=CountVectorizer(stop_words=COMMENT_STOPWORDS, ngram_range=(1, 2), min_df=5),
        representation_model={"KeyBERT": KeyBERTInspired()} if with_keybert else None,
        min_topic_size=MIN_TOPIC_SIZE,
        calculate_probabilities=False,
    )


log.info("Addestramento BERTopic su %d commenti...", len(train_docs))
topic_model = make_topic_model(TOPIC_SEED)
train_topics, _ = topic_model.fit_transform(train_docs, train_embeddings)
train_topics = np.array(train_topics)
log.info("Topic: %d, outlier nel campione: %.1f%%", len(set(train_topics)) - 1, (train_topics == -1).mean() * 100)

log.info("Assegnazione dei topic a %d commenti...", len(eligible_docs))
eligible_topics, _ = topic_model.transform(eligible_docs, eligible_embeddings)
eligible["topic"] = np.array(eligible_topics)

### 6.6 Outlier

Prima esecuzione: gli outlier (topic −1, commenti che HDBSCAN non assegna a nessun cluster abbastanza denso) **restano tali**. Qui si misura quanti sono, in quali periodi e a quali lunghezze. La riga "10-11 parole" serve a verificare la soglia di §6.2.

In [ ]:
eligible["outlier"] = eligible["topic"] == -1
length_bins = pd.cut(eligible["word_count_clean"], bins=[9, 11, 14, 19, 29, 49, 99, 10_000],
                     labels=["10-11", "12-14", "15-19", "20-29", "30-49", "50-99", "100+"])
display(pd.Series({
    "commenti con topic assegnato": int((~eligible["outlier"]).sum()),
    "outlier": int(eligible["outlier"].sum()),
    "outlier %": round(eligible["outlier"].mean() * 100, 1),
    "outlier nel campione di addestramento %": round((train_topics == -1).mean() * 100, 1),
}))
display(eligible.groupby("period", observed=True)["outlier"].mean().mul(100).round(1).to_frame("outlier %").T)
display(eligible.groupby(length_bins, observed=True)["outlier"].agg(["size", "mean"])
        .rename(columns={"size": "commenti", "mean": "outlier %"}).assign(**{"outlier %": lambda x: (x["outlier %"] * 100).round(1)}).T)
eligible.groupby("is_reply")["outlier"].mean().mul(100).round(1).rename({False: "commenti", True: "risposte"}).to_frame("outlier %").T

### 6.7 Topic, parole caratterizzanti e periodi

Tabella dei topic: dimensione nel campione e su tutti i commenti, parole c-TF-IDF e KeyBERT. Poi la quota di ogni topic nei commenti di ciascun periodo (outlier esclusi dal denominatore, colonne che sommano a 100) e due commenti rappresentativi per topic (troncati).

I documenti rappresentativi sono testi di lettori reali: si leggono qui per interpretare i topic, ma non vanno citati alla lettera nel report.

In [ ]:
info = topic_model.get_topic_info()
info = info[info["Topic"] != -1].copy()
info["parole_ctfidf"] = info["Topic"].map(lambda t: ", ".join(w for w, _ in topic_model.get_topic(t)[:8]))
info["parole_keybert"] = info["Topic"].map(lambda t: ", ".join(w for w, _ in topic_model.get_topic(t, full=True)["KeyBERT"][:8]))
info["commenti_campione"] = info["Count"]
info["commenti_tutti"] = info["Topic"].map(eligible["topic"].value_counts()).fillna(0).astype(int)
topic_info = info[["Topic", "commenti_campione", "commenti_tutti", "parole_ctfidf", "parole_keybert"]].rename(columns={"Topic": "topic"})
topic_info.to_csv(OUT_COMMENT_TOPIC_INFO, index=False)
log.info("Scritto %s (%d topic)", OUT_COMMENT_TOPIC_INFO.relative_to(BASE_DIR), len(topic_info))

eligible[["comment_id", "article_id", "period", "topic"]].to_parquet(OUT_COMMENT_TOPICS, index=False)
log.info("Scritto %s", OUT_COMMENT_TOPICS.relative_to(BASE_DIR))

with pd.option_context("display.max_colwidth", 90, "display.max_rows", 200):
    display(topic_info)

In [ ]:
assigned = eligible[~eligible["outlier"]]
topic_period = pd.crosstab(assigned["topic"], assigned["period"], normalize="columns").mul(100).reindex(columns=period_labels)
labels = topic_info.set_index("topic")["parole_ctfidf"].str.split(", ").str[:3].str.join(", ")
topic_period.index = [f"{t}: {labels.get(t, '')}" for t in topic_period.index]

fig, ax = plt.subplots(figsize=(8, 0.28 * len(topic_period) + 1.5))
im = ax.imshow(topic_period.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(period_labels)), period_labels)
ax.set_yticks(range(len(topic_period)), topic_period.index, fontsize=7)
ax.grid(False)
for (r, c), v in np.ndenumerate(topic_period.values):
    if v >= 1:
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=6, color="white" if v > topic_period.values.max() / 2 else "#333")
fig.colorbar(im, ax=ax, label="% dei commenti del periodo con topic")
ax.set_title("Topic dei commenti per periodo (colonne normalizzate)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "comments_topics_by_period.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Due commenti rappresentativi per topic (dal campione di addestramento), troncati
for t in topic_info["topic"].head(40):
    docs = topic_model.get_representative_docs(t)[:2]
    print(f"\n[{t}] {labels.get(t, '')}")
    for d in docs:
        print("   -", d[:160])

### 6.8 Stabilità rapida sui seed

Il testo breve del Guardian era instabile tra seed (AMI 0,39), gli articoli NYT no (0,80). I commenti sono testi brevi: prima di interpretare i topic si controlla che non dipendano dal seed di UMAP. Stessa pipeline sullo stesso campione con 4 seed (KeyBERT omesso: non cambia l'assegnazione), confronto a coppie con l'**Adjusted Mutual Information** (1 = stesse partizioni, 0 = indipendenti): su tutti i commenti del campione e sul solo nucleo (commenti non outlier in entrambe le esecuzioni). Collasso = un topic con più del 50% del campione.

In [ ]:
from itertools import combinations
from sklearn.metrics import adjusted_mutual_info_score

STABILITY_SEEDS = [42, 0, 1, 7]
seed_topics = {TOPIC_SEED: train_topics}
for seed in STABILITY_SEEDS:
    if seed not in seed_topics:
        log.info("Seed %d...", seed)
        model = make_topic_model(seed, with_keybert=False)
        seed_topics[seed] = np.array(model.fit_transform(train_docs, train_embeddings)[0])

seed_table = pd.DataFrame([
    {
        "seed": seed,
        "topic": len(set(t) - {-1}),
        "outlier %": round((t == -1).mean() * 100, 1),
        "topic più grande %": round(pd.Series(t[t != -1]).value_counts().max() / len(t) * 100, 1),
    }
    for seed, t in seed_topics.items()
]).set_index("seed")
display(seed_table)

ami_all, ami_core = [], []
for a, b in combinations(STABILITY_SEEDS, 2):
    ta, tb = seed_topics[a], seed_topics[b]
    ami_all.append(adjusted_mutual_info_score(ta, tb))
    core = (ta != -1) & (tb != -1)
    ami_core.append(adjusted_mutual_info_score(ta[core], tb[core]))
print(f"Collassi (un topic > 50%): {(seed_table['topic più grande %'] > 50).sum()}/{len(seed_table)}")
print(f"AMI tutti i commenti: media {np.mean(ami_all):.3f} (min {min(ami_all):.3f})")
print(f"AMI nucleo:           media {np.mean(ami_core):.3f} (min {min(ami_core):.3f})")

### 6.9 Cosa dicono i risultati (prima esecuzione)

- **56 topic**, tutti leggibili: dall'operato degli agenti (topic 0, il più grande: polizia, agenti, maschere) a lavoro e salari, famiglie e bambini, detenzione e prigioni private, asilo, deportazioni di Obama, proteste di Portland, Corte Suprema, muro, fede cristiana, bilancio, Guardia Nazionale, critica al NYT, fino a temi di nicchia (E-Verify, Statua della Libertà, persone transgender in detenzione).
- **Stabili**: 56-61 topic su 4 seed, nessun collasso, AMI 0,80 su tutti i commenti e 0,98 sul nucleo non outlier. È lo stesso livello degli articoli NYT (0,80) e molto sopra il testo breve del Guardian (0,39): sui commenti BERTopic dà partizioni affidabili.
- **Outlier al 53,7%** sui commenti assegnati con `transform` (50,3% nel campione di addestramento), uniformi tra periodi (50-55%) e tra commenti e risposte (53-54%). Più alti nei commenti brevi (60-62% sotto le 15 parole) ma non trascurabili nemmeno in quelli lunghi (52-54%): molti commenti mescolano più temi o sono opinioni generali senza un tema dominante. Sono il punto da affrontare nella seconda esecuzione (riassegnazione per similarità di embedding o c-TF-IDF, come `reduce_outliers` nel notebook 04).
- **Sovrapposizioni da rivedere**: alcuni topic sono varianti dello stesso tema (11 e 17 democratici / repubblicani; 22 e 26 critiche al NYT; 15 e 29 Biden). Si possono accorpare con `reduce_topics` o `merge_topics` dopo aver deciso come trattare gli outlier.
- **Periodi** (heatmap): detenzione e prigioni al 14% nel 2003-2016; bambini e famiglie al 14% nel 2017-2020 (separazione delle famiglie), con il muro al 4%; asilo all'11% e Russia/Ucraina al 7% nel 2021-2024; nel 2025-presente l'operato degli agenti sale al 23% (dal 4-11%) e le proteste all'8%. Lavoro e salari è l'unico tema costante (6-11% in tutti i periodi).
- Le quote per periodo (heatmap) sono calcolate sui commenti con topic: con oltre metà di outlier vanno lette come composizione della parte classificata, non di tutti i commenti.

## 7. Utenti

Quanto la conversazione è dominata da pochi commentatori abituali, e se il pubblico del 2025-presente è nuovo o è lo stesso degli anni precedenti.

In [ ]:
per_user = comments.groupby("user_key").agg(
    commenti=("comment_id", "size"),
    articoli=("article_id", "nunique"),
)
pd.Series({
    "utenti": len(per_user),
    "mediana commenti/utente": per_user["commenti"].median(),
    "utenti con un solo commento %": round((per_user["commenti"] == 1).mean() * 100, 1),
    "gini commenti/utente": round(gini(per_user["commenti"]), 3),
    "quota commenti top 1% utenti": round(top_share(per_user["commenti"], 0.01), 3),
    "quota commenti top 10% utenti": round(top_share(per_user["commenti"], 0.10), 3),
})

In [ ]:
# Ricambio del pubblico: per ogni periodo, quota di utenti gia' attivi in un periodo precedente
period_labels = [label for label, _, _ in get_periods(int(comments["year"].max()))]
users_by_period = {p: set(comments.loc[comments["period"] == p, "user_key"]) for p in period_labels}

rows, seen = [], set()
for p in period_labels:
    users = users_by_period[p]
    returning = users & seen
    rows.append({
        "periodo": p,
        "utenti": len(users),
        "gia_attivi_prima": len(returning),
        "gia_attivi_prima_pct": round(len(returning) / len(users) * 100, 1) if users else np.nan,
        "commenti_di_utenti_gia_attivi_pct": round(
            comments.loc[(comments["period"] == p) & comments["user_key"].isin(returning)].shape[0]
            / max(1, (comments["period"] == p).sum()) * 100, 1
        ),
    })
    seen |= users
pd.DataFrame(rows)

## 8. Note per il report

- I commenti coprono solo gli articoli in cui il NYT li ha aperti e solo i commenti approvati dalla moderazione: non sono una reazione non mediata come Telegram.
- Fonte non ufficiale (endpoint interno del sito, vedi `docs/nyt_comments.md`).
- Conflittualità = misure strutturali (catene, botta e risposta), non disaccordo testuale; analizzabile solo dal 2018-04 (prima il NYT non registrava risposte a risposte).
- Burst di Kleinberg calcolati per periodo: intensità non confrontabile in assoluto tra periodi; volume poco informativo nei periodi con pochi articoli commentati (2003-2016, 2021-2024).
- NYT Picks concentrati nel 2017-2020 (23 nel 2025-presente): il confronto Picks / più raccomandati descrive soprattutto la scelta editoriale dell'era Trump I. NER `en_core_web_sm` su testo informale dei commenti, non validata a mano.
- Prossimo passo: sentiment dei commenti, con lo stesso modello usato per gli articoli, per confrontare lettori e giornale sugli stessi articoli.